# RAG Pipeline for Marketing Asset Generation
## INFO 290 Group 5 — Integrated Notebook
**Members:** Ryuichi Ikeda, Priya Charagondla, Shivani Parikh, Nseke Ngilbus

### Structure
| Section | Description |
|---------|-------------|
| 0 | Setup: dependencies, credentials, repo clone |
| 1 | Data: load TMDB corpus + genre mapping |
| 2 | RAG corpus: embed + FAISS index |
| 3 | Generator: Mistral-7B-Instruct-v0.2 (4-bit NF4) |
| 4 | Visual grounding: Gemini 2.5 Flash poster captioner |
| 4b | [Archive] Local VLM experiments (vit-gpt2, BLIP-2, Groq) |
| 5 | Offline poster caption batch processor |
| 6 | Core pipeline: retrieve → generate (V1 / V2) |
| 7 | LLM-as-Judge evaluation |
| 8 | Experiment: Idea 1 — Psychological Thriller |


## Section 0: Setup

In [1]:
# Cell 0-A: Install dependencies
!pip -q install "transformers>=4.41" "accelerate>=0.30" bitsandbytes \
                 sentence-transformers faiss-cpu pandas numpy requests tqdm \
                 google-generativeai openai
!pip install -q google-genai

In [2]:
# Cell 0-B: Imports & credentials
import os
import json
import re
import time
import base64
from io import BytesIO
from pprint import pprint

import numpy as np
import pandas as pd
import requests
import torch
import faiss
from tqdm import tqdm
from PIL import Image
from google.colab import userdata

# API credentials (stored in Colab Secrets)
TMDB_BEARER  = userdata.get("TMDB_BEARER")
HF_TOKEN     = userdata.get("HF_TOKEN")
GH_TOKEN     = userdata.get("GITHUB_TOKEN")
GEMINI_API_KEY  = userdata.get("GEMINI_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✅ Imports done")

✅ Imports done


In [3]:
# Cell 0-C: Clone repo
REPO_ROOT = "ryuichi-github-info290-2026s-group5"
if os.path.exists("ryuichi-github-info290-2026s-group5"):
    !git -C ryuichi-github-info290-2026s-group5 pull
else:
    !git clone https://{GH_TOKEN}@github.com/ryuichi-github/ryuichi-github-info290-2026s-group5.git

Already up to date.


In [4]:
# Cell 0-D: Git config for caption branch
import subprocess

# Set git identity
subprocess.run(["git", "-C", REPO_ROOT, "config", "user.email", "i.ryuichi@berkeley.edu"], check=True)
subprocess.run(["git", "-C", REPO_ROOT, "config", "user.name", "Ryuichi Ikeda"], check=True)

# Embed token in remote URL
subprocess.run([
    "git", "-C", REPO_ROOT, "remote", "set-url", "origin",
    f"https://{GH_TOKEN}@github.com/ryuichi-github/ryuichi-github-info290-2026s-group5.git"
], check=True)

# Create and switch to caption-batch branch
result = subprocess.run(
    ["git", "-C", REPO_ROOT, "branch", "--show-current"],
    capture_output=True, text=True
)
current = result.stdout.strip()

if current != "caption-batch":
    r = subprocess.run(
        ["git", "-C", REPO_ROOT, "checkout", "caption-batch"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        subprocess.run(["git", "-C", REPO_ROOT, "checkout", "-b", "caption-batch"], check=True)
        print("Created branch: caption-batch")
    else:
        print("Switched to existing branch: caption-batch")
else:
    print("Already on caption-batch")

# Verify
r = subprocess.run(["git", "-C", REPO_ROOT, "branch", "--show-current"], capture_output=True, text=True)
print(f"Current branch: {r.stdout.strip()}")
print("✅ Git config done")

Already on caption-batch
Current branch: caption-batch
✅ Git config done


In [5]:
# # Manual git commit and push
# import subprocess

# def git_push(repo_root: str, message: str):
#     try:
#         r_add = subprocess.run(
#             ["git", "-C", repo_root, "add", "tmdb-fetch/poster_captions.csv"],
#             capture_output=True, text=True
#         )
#         print(f"git add: {r_add.returncode}")

#         r_commit = subprocess.run(
#             ["git", "-C", repo_root, "commit", "-m", message],
#             capture_output=True, text=True
#         )
#         print(f"git commit: {r_commit.returncode}")
#         print(r_commit.stdout)
#         print(r_commit.stderr)

#         r_push = subprocess.run(
#             ["git", "-C", repo_root, "push", "--set-upstream", "origin", "caption-batch"],
#             capture_output=True, text=True
#         )
#         print(f"git push: {r_push.returncode}")
#         print(r_push.stdout)
#         print(r_push.stderr)

#     except Exception as e:
#         print(f"Error: {e}")

# git_push(REPO_ROOT, "test: manual push from colab")

## Section 1: Data — Load TMDB Corpus

In [6]:
# Cell 1-A: Load dataset

file_path = os.path.join(REPO_ROOT, "tmdb-fetch/tmdb_movies.csv")

if os.path.exists(file_path):
    movies_df = pd.read_csv(file_path)
    print(f"✅ Loaded {len(movies_df)} movies")
    print("Columns:", movies_df.columns.tolist())
else:
    raise FileNotFoundError(f"{file_path} not found. Check repo clone.")

✅ Loaded 4816 movies
Columns: ['id', 'title', 'tagline', 'overview', 'release_date', 'genre_ids', 'genre_names', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'backdrop_path', 'runtime', 'original_language', 'status', 'revenue', 'budget', 'belongs_to_collection']


In [7]:
# Cell 1-B: TMDB API helper + genre mapping
def tmdb_get(url, params=None):
    headers = {
        "Authorization": f"Bearer {TMDB_BEARER}",
        "Content-Type": "application/json;charset=utf-8"
    }
    r = requests.get(url, headers=headers, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# Verify API connection
tmdb_get("https://api.themoviedb.org/3/authentication")

# Genre id <-> name mapping
genre_data = tmdb_get(
    "https://api.themoviedb.org/3/genre/movie/list",
    params={"language": "en-US"}
)
GENRE_ID_TO_NAME = {g["id"]: g["name"] for g in genre_data.get("genres", [])}
GENRE_NAME_TO_ID = {v.lower(): k for k, v in GENRE_ID_TO_NAME.items()}

print(f"✅ Genres loaded: {len(GENRE_ID_TO_NAME)}")

✅ Genres loaded: 19


In [8]:
# Cell 1-C: Dataset subgroup analysis (genre distribution)
# Required for final report per professor feedback

# Parse genre_ids from string representation
corpus_preview = movies_df[
    movies_df["tagline"].notna() & (movies_df["tagline"].str.strip() != "")
].copy()

genre_counts = {}
for gids_str in corpus_preview["genre_ids"].dropna():
    try:
        for gid in eval(gids_str):
            name = GENRE_ID_TO_NAME.get(gid, f"id:{gid}")
            genre_counts[name] = genre_counts.get(name, 0) + 1
    except Exception:
        pass

genre_series = pd.Series(genre_counts).sort_values(ascending=False)
print("Genre distribution in corpus (movies with taglines):")
print(genre_series.to_string())
print(f"\nTotal corpus size: {len(corpus_preview)}")
print("\nNote: one movie can belong to multiple genres.")

Genre distribution in corpus (movies with taglines):
Drama              1845
Action             1414
Thriller           1366
Comedy             1221
Adventure           939
Crime               751
Science Fiction     641
Horror              632
Romance             632
Fantasy             558
Family              506
Animation           389
Mystery             381
History             230
War                 127
Music               102
Western              70
TV Movie             32
Documentary          19

Total corpus size: 4312

Note: one movie can belong to multiple genres.


## Section 2: RAG Corpus — Embed + FAISS Index

In [9]:
# Cell 2: Build RAG corpus
# Corpus  : movies with non-empty taglines (4,312 of 4,816)
# Embedding: title + tagline + overview → Sentence-BERT (all-MiniLM-L6-v2, 384-dim)
# Index   : FAISS IndexFlatIP (inner product on L2-normalised vectors = cosine similarity)

from sentence_transformers import SentenceTransformer

corpus_df = movies_df[
    movies_df["tagline"].notna() & (movies_df["tagline"].str.strip() != "")
].copy().reset_index(drop=True)

print(f"Total movies : {len(movies_df)}")
print(f"Corpus size  : {len(corpus_df)} ({len(corpus_df)/len(movies_df)*100:.1f}%)")
print(f"Excluded     : {len(movies_df) - len(corpus_df)} (missing tagline)")

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_texts = (
    corpus_df["title"] + " " + corpus_df["tagline"] + " " + corpus_df["overview"]
).tolist()

embeddings = embed_model.encode(
    corpus_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print(f"✅ Embeddings : {embeddings.shape}")
print(f"✅ FAISS index: {faiss_index.ntotal} vectors")

Total movies : 4816
Corpus size  : 4312 (89.5%)
Excluded     : 504 (missing tagline)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/135 [00:00<?, ?it/s]

✅ Embeddings : (4312, 384)
✅ FAISS index: 4312 vectors


## Section 3: Generator — Mistral-7B-Instruct-v0.2 (4-bit NF4)

In [10]:
# Cell 3: Load LLM
# Model  : Mistral-7B-Instruct-v0.2
# Quant  : 4-bit NF4 (bitsandbytes) — fits in T4 16GB VRAM

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
)

print("✅ Model loaded on:", model.device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded on: cuda:0


## Section 4: Visual Grounding — Gemini 2.5 Flash (Adopted)

**Decision:** Gemini 2.5 Flash via API is the adopted captioner.  
All other VLM experiments (vit-gpt2, BLIP-2, Groq Llama 4 Scout) are archived in Section 4b.

**6-axis caption structure:**
1. Composition (subject position, angle, negative space)
2. Lighting & shadows (source direction, silhouette)
3. Color palette (dominant tone, accent colors)
4. Emotional tone (unease, isolation, tension...)
5. Typography relationship (text position, image interaction)
6. Iconic visual symbol (one defining object)


In [11]:
# Cell 4-A: Gemini 2.5 Flash setup
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-2.5-flash")

TMDB_IMAGE_BASE = "https://image.tmdb.org/t/p/w780"  # w780 keeps 2:3 ratio without cropping

POSTER_PROMPT = """Analyze this movie poster and write a detailed description in flowing prose.
Do not use bullet points, numbers, or lists.
Do not invent, assume, or hallucinate any details not visible in the image.

Your description must cover:
(1) composition and positioning of subjects,
(2) direction and quality of lighting and shadows,
(3) overall color palette including dominant and accent colors,
(4) emotional mood and atmosphere,
(5) typography including text position and interaction with subjects,
(6) one key iconic visual symbol that defines the poster.

Be specific, vivid, and literal. This description will be used by a designer to
recreate the visual style for a new movie poster."""

print("✅ Gemini 2.5 Flash ready")

✅ Gemini 2.5 Flash ready


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [12]:
# Cell 4-B: describe_poster_gemini() — single poster captioner
def describe_poster_gemini(poster_path: str) -> str:
    """
    Fetch a TMDB poster and return a 6-axis prose caption via Gemini 2.5 Flash.

    Parameters
    ----------
    poster_path : str
        TMDB poster_path field (e.g. '/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg')

    Returns
    -------
    str
        Prose caption, or empty string on failure.
    """
    if not poster_path or pd.isna(poster_path):
        return ""
    try:
        url = TMDB_IMAGE_BASE + poster_path
        r = requests.get(url, timeout=15, stream=True)
        img_bytes = b"".join(chunk for chunk in r.iter_content(chunk_size=4096) if chunk)
        img = Image.open(BytesIO(img_bytes)).convert("RGB")
        response = gemini.generate_content([img, POSTER_PROMPT])
        return response.text.strip()
    except Exception as e:
        print(f"Poster description failed for {poster_path}: {e}")
        return ""

print("✅ describe_poster_gemini ready")

✅ describe_poster_gemini ready


In [13]:
# Cell 4-C: Single-poster test (Take Shelter)
# test_path = "/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg"
# caption = describe_poster_gemini(test_path)
# print(f"Caption:\n{caption}")

## Section 4b: [Archive] Local VLM Experiments

These cells document the VLM comparison that led to adopting Gemini 2.5 Flash.  
**Do not run in production** — these require GPU memory alongside Mistral-7B.

| Model | Verdict | Reason |
|-------|---------|--------|
| vit-gpt2 (200MB, CPU) | Rejected | No prompt support, object misrecognition |
| BLIP-2 zero-shot (3.9B, CPU) | Rejected | Reads poster text only |
| BLIP-2 Q&A prompt (3.9B, CPU) | Insufficient | Context collapse ("dark and moody" for all) |
| Groq Llama 4 Scout (API) | Rejected | Hallucinated fictional titles in generation |
| **Gemini 2.5 Flash (API)** | **Adopted** | Accurate 6-axis prose, no hallucination |


In [14]:
# [ARCHIVE] vit-gpt2 experiment — do not run in production
# from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer as VitTokenizer
# vit_model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning").to("cpu")
# vit_processor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
# vit_tokenizer = VitTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
# Result on Take Shelter: "man standing in front of large group of birds" ← rejected

In [15]:
# [ARCHIVE] BLIP-2 experiment — do not run in production
# from transformers import Blip2Processor, Blip2ForConditionalGeneration
# blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
# blip_model = Blip2ForConditionalGeneration.from_pretrained(
#     "Salesforce/blip2-flan-t5-xl", torch_dtype=torch.float32
# ).to("cpu")
# Zero-shot result: "the poster is dark and moody" ← context collapse, rejected
# Q&A result: "a man and woman standing in front of a stormy sky" ← too shallow

In [16]:
# [ARCHIVE] Groq Llama 4 Scout — do not use for generation pipeline
# Rejected because it hallucinated a fictional title "THE ISOLATED" in poster_art_direction output.
# The poster captioning quality itself was acceptable but the downstream contamination was disqualifying.
#
# from groq import Groq
# GROQ_API_KEY = userdata.get("GROQ_API_KEY")
# groq_client = Groq(api_key=GROQ_API_KEY)

## Section 5: Offline Poster Caption Batch Processor

Pre-process all 4,312 posters with Gemini 2.5 Flash and save captions to CSV.  
At inference time, captions are looked up by film ID — no runtime API calls needed.

**Rationale:** Retrieval axis is narrative/marketing similarity (textual), not visual similarity.  
Poster captions are supplementary context for generation, not a search dimension.

**Cost estimate:** ~$5 at Gemini 2.5 Flash paid tier for 4,312 images.


In [17]:
# Cell 5-A: Offline batch captioner
# Saves after every single record. Safe to interrupt and resume anytime.

CAPTION_OUTPUT = os.path.join(REPO_ROOT, "tmdb-fetch/poster_captions.csv")

import subprocess

def _git_push(repo_root: str, message: str):
    """Commit and push latest CSV to GitHub."""
    try:
        subprocess.run(["git", "-C", repo_root, "add", "tmdb-fetch/poster_captions.csv"], check=True)
        subprocess.run(["git", "-C", repo_root, "commit", "-m", message], check=True)
        subprocess.run(["git", "-C", repo_root, "push"], check=True)
        print(f"  Pushed to GitHub: {message}")
    except subprocess.CalledProcessError as e:
        print(f"  Git push failed: {e}")

def batch_caption_posters(
    df: pd.DataFrame,
    output_path: str,
    sleep_between: float = 0.5,
    push_interval: int = 50,
    repo_root: str = None,
) -> pd.DataFrame:
    """
    Generate Gemini 2.5 Flash captions for all posters in df.

    Appends one row at a time and pushes to GitHub every push_interval records.
    Safe to interrupt — resumes from last saved state on restart.

    Parameters
    ----------
    df            : DataFrame with columns [id, title, poster_path]
    output_path   : path to save/resume the caption CSV
    sleep_between : seconds between API calls
    push_interval : commit and push to GitHub every N new records
    repo_root     : path to git repo root (required for push)
    """
    if os.path.exists(output_path):
        done_ids = set(pd.read_csv(output_path, usecols=["id"])["id"].tolist())
        print(f"Resuming: {len(done_ids)} done, {len(df) - len(done_ids)} remaining.")
    else:
        done_ids = set()
        pd.DataFrame(columns=["id", "title", "poster_path", "poster_caption"]
                     ).to_csv(output_path, index=False)
        print(f"Starting fresh: {len(df)} posters to process.")

    remaining = df[~df["id"].isin(done_ids)]

    for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining)), 1):
        caption = describe_poster_gemini(row.get("poster_path", None))
        pd.DataFrame([{
            "id": row["id"],
            "title": row["title"],
            "poster_path": row.get("poster_path", ""),
            "poster_caption": caption,
        }]).to_csv(output_path, mode="a", header=False, index=False)

        total_done = len(done_ids) + i
        if i % 10 == 0:
            print(f"  [{total_done}/{len(df)}] {row['title']} → {caption[:60]}...")

        # Push to GitHub every push_interval records
        if i % push_interval == 0 and repo_root:
            _git_push(repo_root, f"caption progress: {total_done}/{len(df)} posters")

        time.sleep(sleep_between)

    # Final push
    if repo_root:
        _git_push(repo_root, f"caption complete: {len(df)} posters")

    result = pd.read_csv(output_path)
    print(f"✅ Done. Total rows saved: {len(result)}")
    return result

print("✅ batch_caption_posters ready")

✅ batch_caption_posters ready


In [18]:
print(type(gemini))
print(GEMINI_API_KEY[:10] if GEMINI_API_KEY else "None")

import google.generativeai as genai

try:
    response = gemini.generate_content("Say hello in one word.")
    print(response.text)
except Exception as e:
    print(f"Error: {e}")

<class 'google.generativeai.generative_models.GenerativeModel'>
AIzaSyCR0l
Hello


In [19]:
# デバッグ: 1枚だけ手動でテスト
# test_row = corpus_df[corpus_df["poster_path"].notna()].iloc[0]
# print(f"Testing: {test_row['title']} ({test_row['poster_path']})")
# result = describe_poster_gemini(test_row["poster_path"])
# print(f"Result: '{result[:100]}'")

In [20]:
# Cell 5-B: Run batch captioning
# WARNING: costs ~$5 in Gemini API credits. Run once and commit the CSV.

# caption_df = batch_caption_posters(
#     df=corpus_df[["id", "title", "poster_path"]],
#     output_path=CAPTION_OUTPUT,
#     push_interval=50,
#     repo_root=REPO_ROOT,
# )
# print(f"✅ Captioning complete. Total rows: {len(caption_df)}")

# To load pre-generated captions:
if os.path.exists(CAPTION_OUTPUT):
    caption_df = pd.read_csv(CAPTION_OUTPUT)
    corpus_df = corpus_df.merge(
        caption_df[["id", "poster_caption"]], on="id", how="left"
    )
    print(f"✅ Captions loaded. Coverage: {corpus_df['poster_caption'].notna().sum()}/{len(corpus_df)}")
else:
    print("poster_captions.csv not found. Run batch captioning first.")

✅ Captions loaded. Coverage: 4288/4312


## Section 6: Core Pipeline — Retrieval, Generation, V1/V2

**Fixed variables across all experiments:**
- Retrieval corpus: 4,312 movies
- k = 5, genre filter applied
- Embedding model: all-MiniLM-L6-v2
- Generator: Mistral-7B-Instruct-v0.2 (4-bit NF4)
- Input format: structured dict (core_premise / thematic_core / negative_constraints)


In [21]:
# Cell 6-A: Genre filter helper
def _normalize_genre_filter(genre_filter):
    """
    Accepts None, list of genre name strings, or list of genre int IDs.
    Returns list of int IDs or None.
    """
    if genre_filter is None:
        return None
    if isinstance(genre_filter, (list, tuple)) and len(genre_filter) > 0:
        if isinstance(genre_filter[0], str):
            ids = [GENRE_NAME_TO_ID[name.lower()] for name in genre_filter
                   if name.lower() in GENRE_NAME_TO_ID]
            return ids if ids else []
        if isinstance(genre_filter[0], int):
            return list(genre_filter)
    raise ValueError("genre_filter must be None, list of genre name strings, or list of genre int IDs.")

In [22]:
# Cell 6-B: Retrieval
def retrieve(query: str, k: int = 5, genre_filter=None) -> pd.DataFrame:
    """
    Retrieve top-k similar movies from the corpus.

    Parameters
    ----------
    query        : free-text query (typically core_premise)
    k            : number of results to return
    genre_filter : list of genre names or IDs; None = no filter

    Returns
    -------
    DataFrame with top-k rows from corpus_df plus a 'score' column.
    Falls back to unfiltered search if no genre-matching candidates exist.
    """
    q = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    gf_ids = _normalize_genre_filter(genre_filter)

    if gf_ids is None:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        return out.reset_index(drop=True)

    gf_ids = set(gf_ids)
    mask = corpus_df["genre_ids"].apply(
        lambda gids: any(gid in gf_ids for gid in eval(gids))
    ).to_numpy()
    candidate_idx = np.where(mask)[0]

    if candidate_idx.size == 0:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        out["note"] = "No genre-filtered candidates; fell back to unfiltered."
        return out.reset_index(drop=True)

    cand_emb = embeddings[candidate_idx]
    scores = cand_emb @ q[0]
    topk_local = np.argsort(-scores)[:k]
    topk_idx = candidate_idx[topk_local]
    out = corpus_df.iloc[topk_idx].copy()
    out["score"] = scores[topk_local]
    return out.reset_index(drop=True)

In [23]:
# Cell 6-C: Reference text builder
def refs_to_text(df: pd.DataFrame, n: int = 5, use_visual: bool = True) -> str:
    """
    Format retrieved movies as a reference string for the prompt.

    Includes tagline + overview (truncated to 200 chars) as style reference.
    Optionally appends pre-computed poster caption if use_visual=True and
    poster_caption column is present.

    Parameters
    ----------
    df         : top-k retrieval result from retrieve()
    n          : number of references to include
    use_visual : whether to include poster_caption (V2 only)
    """
    lines = []
    for _, r in df.head(n).iterrows():
        if pd.notna(r["tagline"]) and r["tagline"].strip():
            ov = (r["overview"] or "")[:200].replace("\n", " ")
            line = f'- {r["title"]}: tagline: "{r["tagline"]}" | overview: "{ov}"'
            if use_visual and "poster_caption" in r and pd.notna(r.get("poster_caption")):
                caption = str(r["poster_caption"]).strip()
                if caption:
                    line += f' | visual style: "{caption[:300]}"'
            lines.append(line)
    return "\n".join(lines)

In [24]:
# Cell 6-D: Text generation
def generate_text(prompt: str, max_new_tokens: int = 320, temperature: float = 0.6) -> str:
    """
    Generate text with Mistral-7B using greedy-ish sampling.

    Parameters
    ----------
    prompt         : full formatted prompt string
    max_new_tokens : generation budget (320 is sufficient for JSON output)
    temperature    : sampling temperature for do_sample generation
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.10,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [25]:
# Cell 6-E: JSON extraction
REQUIRED_KEYS = {"overview", "tagline", "poster_art_direction"}

def extract_best_json(text: str) -> dict:
    """
    Extract the best-matching JSON object from raw model output.

    Scans the full text for JSON objects and scores each candidate by
    presence of required keys and plausible field lengths.
    Raises ValueError if no JSON object is found.
    """
    decoder = json.JSONDecoder()
    objs = []
    i = 0
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            objs.append(obj)
            i = start + end
        except json.JSONDecodeError:
            i = start + 1

    if not objs:
        raise ValueError("No JSON object found in model output.")

    def score(obj):
        if not isinstance(obj, dict):
            return -10
        if not REQUIRED_KEYS.issubset(set(obj.keys())):
            return -5
        s = 0
        tagline  = obj.get("tagline", "")
        poster   = obj.get("poster_art_direction", "")
        overview = obj.get("overview", "")
        if isinstance(tagline, str) and len(tagline.strip()) > 0:
            s += 2
            if len(tagline.split()) <= 14:
                s += 1
        if isinstance(poster, str) and len(poster.strip()) > 20:
            s += 2
        if isinstance(overview, str) and len(overview.strip()) > 40:
            s += 2
        return s

    return max(objs, key=score)

In [26]:
# Cell 6-F: Prompt builder
def build_prompt(movie_input: dict, refs: str) -> str:
    """
    Build the generation prompt from structured input and retrieved references.

    movie_input keys: core_premise, thematic_core, negative_constraints
    refs            : formatted reference string from refs_to_text()
    """
    premise   = movie_input.get("core_premise", "")
    theme     = movie_input.get("thematic_core", "")
    negatives = movie_input.get("negative_constraints", "")

    return f"""You are a movie marketing generator.

Generate marketing assets for the movie described below.

OUTPUT: ONE JSON object with EXACTLY these keys:
- overview (<= 80 words, compelling promo synopsis)
- tagline (<= 12 words)
- poster_art_direction (<= 60 words)

JSON FORMAT:
{{"overview": "", "tagline": "", "poster_art_direction": ""}}

MOVIE DETAILS:
Core Premise: {premise}
Thematic Core: {theme}
IMPORTANT - Do NOT do the following: {negatives}

REFERENCES from similar successful movies:
⚠ STRICT RULE: Use these for WRITING STYLE, TONE, and SENTENCE RHYTHM ONLY.
Do NOT borrow any character names, plot events, locations, or story elements.
Every word of your output must describe ONLY the movie in MOVIE DETAILS above.
If you find yourself writing anything not in MOVIE DETAILS, stop and rewrite.
{refs}

CONSTRAINTS:
- Output ONLY the JSON object (no markdown, no backticks, no commentary).
- tagline must be original and specific to this movie, not generic.
- overview must be written as compelling promo copy, not a plot summary.
- End immediately after the final '}}'.

Now output the JSON:
"""

print("✅ build_prompt ready")

✅ build_prompt ready


In [27]:
# Cell 6-G: V1 / V2 runner
def run_v1_v2(movie_input: dict, k: int = 5, genre_filter=None, temperature: float = 0.6):
    """
    Run both V1 (zero-shot) and V2 (RAG + visual grounding) for a given input.

    V1: no retrieval, prompt receives refs="(none)"
    V2: retrieves top-k, includes poster_caption if available (use_visual=True)

    Returns
    -------
    (v1_output, v2_output, topk_df, refs_string)
    """
    query = movie_input["core_premise"]

    # V1: zero-shot baseline
    p1 = build_prompt(movie_input, refs="(none)")
    t1 = generate_text(p1, temperature=temperature)
    j1 = extract_best_json(t1)

    # V2: RAG + visual grounding
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)
    p2 = build_prompt(movie_input, refs=refs)
    t2 = generate_text(p2, temperature=temperature)
    j2 = extract_best_json(t2)

    return j1, j2, topk, refs

print("✅ run_v1_v2 ready")

✅ run_v1_v2 ready


In [28]:
# Cell 6-H: V3 Agentic Loop runner
from openai import OpenAI


def run_v3(
    movie_input: dict,
    k: int = 5,
    genre_filter=None,
    temperature: float = 0.6,
    max_iter: int = 3,
    judge_client: OpenAI = None,
) -> tuple[dict, list]:
    """
    RAG generation with iterative refinement: the judge's weakest-dimension
    feedback is injected into the next prompt (same RAG refs as V2).
    """
    if judge_client is None:
        judge_client = OpenAI()

    # Same dimension order as Cell 7-A (DIMENSIONS = list(RUBRIC.keys())).
    try:
        dim_order = DIMENSIONS
    except NameError:
        dim_order = [
            "narrative_fidelity",
            "genre_alignment",
            "visual_specificity",
            "creative_specificity",
            "output_format_validity",
        ]

    query = movie_input["core_premise"]
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)

    feedback = ""
    history = []
    best_assets = None

    for iteration in range(1, max_iter + 1):
        base = build_prompt(movie_input, refs=refs)
        if feedback:
            prompt = (
                base
                + "\n\nPREVIOUS FEEDBACK:\n"
                + feedback
                + "\nAddress the above before regenerating.\n"
            )
        else:
            prompt = base

        raw = generate_text(prompt, temperature=temperature)

        try:
            assets = extract_best_json(raw)
        except (ValueError, TypeError, json.JSONDecodeError):
            # Keep prior best_assets; give the model concrete repair instructions.
            tagline = "(parse failed)"
            print(f"[v3 iter {iteration}] tagline={tagline!r} average=n/a")
            history.append(
                {
                    "iter": iteration,
                    "assets": None,
                    "scores": None,
                    "average": None,
                    "passed": False,
                }
            )
            feedback = (
                "Your previous output was not valid JSON with the required keys. "
                "Output exactly one JSON object with keys overview, tagline, poster_art_direction."
            )
            continue

        # Last successful parse wins for the return value.
        best_assets = assets

        result = llm_as_judge(
            movie_input,
            assets,
            client=judge_client,
            verbose=False,
        )
        scores = result.get("scores") or {}
        avg = result.get("average")

        dim_scores = []
        for dim in dim_order:
            entry = scores.get(dim, {}) if isinstance(scores, dict) else {}
            s = entry.get("score")
            if isinstance(s, (int, float)):
                dim_scores.append(s)

        passed = (
            len(dim_scores) == len(dim_order)
            and all(s >= 3 for s in dim_scores)
        )

        history.append(
            {
                "iter": iteration,
                "assets": assets,
                "scores": scores,
                "average": avg,
                "passed": passed,
            }
        )

        tagline = (assets.get("tagline") or "")[:120]
        avg_str = f"{avg:.2f}" if isinstance(avg, (int, float)) else "n/a"
        print(f"[v3 iter {iteration}] tagline={tagline!r} average={avg_str}")

        if passed:
            break

        # Next round: feedback = reason for the lowest-scoring dimension (ties broken by dim_order).
        low_dim = None
        low_score = float("inf")
        for dim in dim_order:
            entry = scores.get(dim, {}) if isinstance(scores, dict) else {}
            s = entry.get("score")
            if isinstance(s, (int, float)) and s < low_score:
                low_score = s
                low_dim = dim

        if low_dim is not None:
            reason = scores.get(low_dim, {}).get("reason", "")
            feedback = reason if reason else f"Improve the {low_dim} dimension."
        else:
            feedback = "Improve the output on all rubric dimensions."

    return best_assets, history


print("✅ run_v3 ready")


✅ run_v3 ready


## Section 7: LLM-as-Judge Evaluation

**Design decisions:**
- Judge model: gpt-4o (different from generator Mistral-7B — avoids self-enhancement bias)
- Evaluation axes: 5 dimensions matching observed failure modes during development
- Scoring: 1–5 integer per dimension + one-sentence reason (for interpretability)
- Batch mode: `evaluate_batch()` for ~200 QA pairs; `summarize_batch()` for aggregated stats

**Reference:** MT-Bench / Chatbot Arena (Zheng et al., 2023) for LLM-as-Judge methodology.


In [29]:
# Cell 7-A: Rubric definition
from openai import OpenAI

JUDGE_MODEL = "gpt-4o"

RUBRIC = {
    "narrative_fidelity": {
        "description": "Output follows intended premise without adding unintended plot elements",
        "5": "No negative_constraints violated. All content anchored strictly to core_premise and thematic_core.",
        "3": "Minor trope drift or one incidental element not in the premise; no invented characters or major additions.",
        "1": "Invented characters, disappearances, villains, or plot elements explicitly excluded by negative_constraints.",
    },
    "genre_alignment": {
        "description": "Output matches the intended genre and tone",
        "5": "Vocabulary, sentence rhythm, and emotional register fully match the target genre.",
        "3": "Mostly aligned but one phrase or image feels like it belongs to a different genre.",
        "1": "Clearly wrong tone — e.g., action-thriller language for a slow psychological piece.",
    },
    "visual_specificity": {
    "5": "Specifies at least 3 of: subject position, lighting direction, "
         "color palette, dominant shadow/silhouette, and one symbolic object. "
         "A designer could start sketching immediately from this description.",
    "3": "Mentions atmosphere and mood but missing 2+ of the above specifics.",
    "1": "Only generic descriptors like 'dark and moody' with no actionable detail.",
},
"creative_specificity": {
    "5": "The tagline contains an unexpected word combination or image that "
         "could not have come from a generic LLM prompt. Memorable on first read.",
    "3": "Competent and on-brief but interchangeable with many other thrillers.",
    "1": "Cliché phrase that appears verbatim in common marketing templates.",
},
    "output_format_validity": {
        "description": "Output follows required structure and word limits",
        "5": "All three fields present. tagline ≤ 12 words, overview ≤ 80 words, poster_art_direction ≤ 60 words.",
        "3": "All fields present but one field slightly exceeds the word limit (≤ 20% over).",
        "1": "A required field is missing, or one field is more than 20% over the word limit.",
    },
}

DIMENSIONS = list(RUBRIC.keys())
print("✅ Rubric defined")

✅ Rubric defined


In [30]:
# Cell 7-B: Prompt builder for judge
def _build_judge_prompt(movie_input: dict, generated_output: dict) -> str:
    rubric_text = ""
    for dim, content in RUBRIC.items():
        rubric_text += f"\n{dim}:\n"
        rubric_text += f"  5 = {content['5']}\n"
        rubric_text += f"  3 = {content['3']}\n"
        rubric_text += f"  1 = {content['1']}\n"

    return f"""You are an expert film marketing evaluator.
Score the following generated marketing assets on 5 dimensions, each on a 1–5 integer scale.

[INPUT CONCEPT]
core_premise: {movie_input.get('core_premise', '')}
thematic_core: {movie_input.get('thematic_core', '')}
negative_constraints: {movie_input.get('negative_constraints', '')}

[GENERATED OUTPUTS]
tagline: {generated_output.get('tagline', '')}
overview: {generated_output.get('overview', '')}
poster_art_direction: {generated_output.get('poster_art_direction', '')}

[RUBRIC]{rubric_text}
Scoring rules:
- Score each dimension independently.
- Assign integer scores only: 1, 2, 3, 4, or 5.
- Write one concise sentence as reason for each score.
- Do NOT copy phrasing from the generated output in your reasons.

Return ONLY valid JSON. No markdown, no preamble. Schema:
{{
  "scores": {{
    "narrative_fidelity":      {{"score": <int>, "reason": "<str>"}},
    "genre_alignment":         {{"score": <int>, "reason": "<str>"}},
    "visual_specificity":      {{"score": <int>, "reason": "<str>"}},
    "creative_specificity":    {{"score": <int>, "reason": "<str>"}},
    "output_format_validity":  {{"score": <int>, "reason": "<str>"}}
  }},
  "average": <float>
}}"""

print("✅ _build_judge_prompt ready")

✅ _build_judge_prompt ready


In [31]:
# Cell 7-C: llm_as_judge() — single evaluation
def llm_as_judge(
    movie_input: dict,
    generated_output: dict,
    client: OpenAI = None,
    max_retries: int = 3,
    retry_delay: float = 2.0,
    verbose: bool = True,
) -> dict:
    """
    Score a single generated output on 5 dimensions using GPT-4o as judge.

    Parameters
    ----------
    movie_input       : structured input dict (core_premise, thematic_core, negative_constraints)
    generated_output  : generated dict (tagline, overview, poster_art_direction)
    client            : OpenAI instance; created from env var if None
    max_retries       : JSON parse retry attempts
    retry_delay       : seconds between retries
    verbose           : print score summary if True

    Returns
    -------
    dict with keys: scores (per dimension), average (float), error (str|None)
    """
    if client is None:
        client = OpenAI()

    prompt = _build_judge_prompt(movie_input, generated_output)

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL,
                max_tokens=1000,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": (
                        "You are a strict film marketing evaluator. "
                        "Return only valid JSON matching the requested schema. "
                        "No markdown, no preamble, no explanation outside the JSON."
                    )},
                    {"role": "user", "content": prompt},
                ],
            )
            raw_text = response.choices[0].message.content.strip()
            raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
            raw_text = re.sub(r"\s*```$", "", raw_text)

            result = json.loads(raw_text)

            # Recompute average to guard against judge arithmetic errors
            scores = result.get("scores", {})
            vals = [v["score"] for v in scores.values() if isinstance(v.get("score"), (int, float))]
            result["average"] = round(sum(vals) / len(vals), 2) if vals else 0.0
            result["error"] = None

            if verbose:
                _print_judge_result(result)
            return result

        except (json.JSONDecodeError, KeyError) as e:
            if attempt < max_retries:
                if verbose:
                    print(f"[judge] attempt {attempt} failed ({e}), retrying in {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                return {
                    "scores": {},
                    "average": 0.0,
                    "error": str(e),
                    "raw": raw_text if "raw_text" in dir() else "",
                }


def _print_judge_result(result: dict) -> None:
    print("\n" + "=" * 52)
    print(f"  average: {result['average']:.2f} / 5.00")
    print("=" * 52)
    for dim in DIMENSIONS:
        entry = result["scores"].get(dim, {})
        score  = entry.get("score", "?")
        reason = entry.get("reason", "")
        bar = "█" * score + "░" * (5 - score) if isinstance(score, int) else ""
        print(f"  {dim:<28} {bar}  {score}/5")
        print(f"    → {reason}")
    print("=" * 52 + "\n")


print("✅ llm_as_judge ready")

✅ llm_as_judge ready


In [32]:
# Cell 7-D: evaluate_batch() + summarize_batch() — for 200 QA pairs
def evaluate_batch(
    records: list,
    client: OpenAI = None,
    sleep_between: float = 0.5,
    verbose: bool = False,
) -> list:
    """
    Evaluate a list of records with llm_as_judge().

    Parameters
    ----------
    records        : list of dicts with keys: movie_input, generated_output, id (optional)
    client         : shared OpenAI client
    sleep_between  : API rate-limit buffer in seconds
    verbose        : print per-record scores if True

    Returns
    -------
    list of dicts: [{id, result}, ...]
    """
    if client is None:
        client = OpenAI()

    results = []
    total = len(records)

    for i, record in enumerate(records, 1):
        record_id = record.get("id", f"record_{i:04d}")
        print(f"[{i}/{total}] evaluating {record_id}...", end=" ", flush=True)

        result = llm_as_judge(
            movie_input=record["movie_input"],
            generated_output=record["generated_output"],
            client=client,
            verbose=verbose,
        )
        avg = result.get("average", 0.0)
        err = result.get("error")
        print(f"avg={avg:.2f}" if not err else f"ERROR: {err}")

        results.append({"id": record_id, "result": result})

        if i < total:
            time.sleep(sleep_between)

    return results


def summarize_batch(batch_results: list) -> dict:
    """
    Aggregate batch evaluation results into per-dimension and overall statistics.

    Returns
    -------
    dict with keys: n, per_dimension (mean/min/max per axis), overall_mean, error_count
    """
    per_dim = {d: [] for d in DIMENSIONS}
    averages = []
    error_count = 0

    for record in batch_results:
        r = record["result"]
        if r.get("error"):
            error_count += 1
            continue
        averages.append(r["average"])
        for dim in DIMENSIONS:
            score = r["scores"].get(dim, {}).get("score")
            if score is not None:
                per_dim[dim].append(score)

    summary = {
        "n": len(batch_results),
        "per_dimension": {},
        "overall_mean": round(sum(averages) / len(averages), 2) if averages else 0.0,
        "error_count": error_count,
    }
    for dim, vals in per_dim.items():
        if vals:
            summary["per_dimension"][dim] = {
                "mean": round(sum(vals) / len(vals), 2),
                "min": min(vals),
                "max": max(vals),
            }
    return summary


print("✅ evaluate_batch and summarize_batch ready")

✅ evaluate_batch and summarize_batch ready


## Section 8: Experiment — Idea 1 (Psychological Thriller)

**Input:** Structured dict with core_premise / thematic_core / negative_constraints  
**Variants compared:** V1 (zero-shot) vs V2 (RAG + Gemini visual grounding)  
**Fixed variables:** k=5, genre_filter=[thriller, horror, mystery], Mistral-7B NF4


In [ ]:
# Cell 8-A: Idea 1 input definition
movie_input = {
    "core_premise": (
        "A man wakes up in a hospital with no memory of his past. "
        "People claiming to be his family and friends begin to appear one by one — "
        "each with a different account of who he was. "
        "As fragments of memory return, he starts to suspect that someone "
        "is trying to control what he remembers."
    ),
    "thematic_core": (
        "Identity is not what you remember — it is what others need you to believe. "
        "The horror of discovering that your own past makes you the monster."
    ),
    "negative_constraints": (
        "Do NOT reveal whether the visitors are real or fabricated until the very end. "
        "Do NOT use action sequences or physical violence as plot drivers. "
        "Do NOT frame memory recovery as heroic — it should feel increasingly terrifying."
    ),
}
print("Input defined.")

Input defined.


In [34]:
# Cell 8-B: Run V1 and V2
v1, v2, topk, refs = run_v1_v2(
    movie_input,
    k=5,
    genre_filter=["thriller", "horror", "mystery"]
)

print("=== Retrieved movies ===")
display(topk[["title", "genre_names", "tagline", "score"]])

print("\n=== References passed to V2 ===")
print(refs)

print("\n=== V1: Zero-Shot ===")
pprint(v1)

print("\n=== V2: RAG + Visual Grounding ===")
pprint(v2)

=== Retrieved movies ===


,title,genre_names,tagline,score
0,Take Shelter,"['Thriller', 'Drama', 'Horror']",Far away from the cruel world.,0.489664
1,Hard Rain,"['Thriller', 'Crime', 'Action']",A simple plan. An instant fortune. Just add wa...,0.457851
2,Cure,"['Crime', 'Thriller', 'Horror', 'Mystery']",Madness. Terror. Murder.,0.427415
3,Regression,"['Horror', 'Mystery', 'Thriller']",Fear always finds its victim,0.415956
4,Delirium,"['Horror', 'Thriller']",It's all in your head,0.386188



=== References passed to V2 ===
- Take Shelter: tagline: "Far away from the cruel world." | overview: "Plagued by a series of apocalyptic visions, a young husband and father questions whether to shelter his family from a coming storm, or from himself." | visual style: "The movie poster presents a deeply unsettling scene, drawing the viewer into an atmosphere of profound dread and anticipation. At the forefront, occupying a significant portion of the lower left and center, stands a man, seen from the chest up, his gaze directed intensely upwards and slightly to the"
- Hard Rain: tagline: "A simple plan. An instant fortune. Just add water." | overview: "An armored car driver tries to elude a gang of thieves while a flood ravages the countryside." | visual style: "The movie poster is an electrifying depiction of extreme peril and desperate flight, visually split into distinct yet interconnected realms of danger. The composition is dominated by a lone male figure, presumably the protagoni

In [35]:
# Cell 8-C: Evaluate V1 and V2 with LLM-as-Judge
judge_client = OpenAI()

print(">>> Evaluating V1")
result_v1 = llm_as_judge(movie_input, v1, client=judge_client)

print(">>> Evaluating V2")
result_v2 = llm_as_judge(movie_input, v2, client=judge_client)

print(f"\nV1 average : {result_v1['average']:.2f}")
print(f"V2 average : {result_v2['average']:.2f}")
print(f"Delta (V2-V1): {result_v2['average'] - result_v1['average']:+.2f}")

>>> Evaluating V1

  average: 4.60 / 5.00
  narrative_fidelity           █████  5/5
    → The assets strictly adhere to the core premise and thematic core without violating constraints.
  genre_alignment              █████  5/5
    → The language and mood perfectly match the psychological thriller genre.
  visual_specificity           ███░░  3/5
    → The description provides atmosphere but lacks detailed visual design elements.
  creative_specificity         █████  5/5
    → The tagline uses a unique and memorable turn of phrase suitable for the genre.
  output_format_validity       █████  5/5
    → All fields are present and within the specified word limits.

>>> Evaluating V2

  average: 4.40 / 5.00
  narrative_fidelity           █████  5/5
    → The content stays focused on paranoia and flooding without violating constraints.
  genre_alignment              █████  5/5
    → The language conveys psychological thriller tension with consistent thematic tone.
  visual_specificity       

In [36]:
# Cell 8-D: Per-dimension comparison table
rows = []
for dim in DIMENSIONS:
    s1 = result_v1["scores"].get(dim, {}).get("score", None)
    s2 = result_v2["scores"].get(dim, {}).get("score", None)
    rows.append({
        "dimension": dim,
        "V1 (zero-shot)": s1,
        "V2 (RAG+visual)": s2,
        "delta": (s2 - s1) if (s1 is not None and s2 is not None) else None,
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)

,dimension,V1 (zero-shot),V2 (RAG+visual),delta
0,narrative_fidelity,5,5,0
1,genre_alignment,5,5,0
2,visual_specificity,3,3,0
3,creative_specificity,5,4,-1
4,output_format_validity,5,5,0


In [37]:
# Cell 8-E: Run V3 on Idea 1 and compare V1 / V2 / V3

judge_client = OpenAI()
v3, v3_history = run_v3(
    movie_input,
    k=5,
    genre_filter=["thriller", "horror", "mystery"],
    judge_client=judge_client,
)

result_v3 = llm_as_judge(movie_input, v3, client=judge_client)

print("=== V3: Final output ===")
pprint(v3)

print("\n=== V3: Per-iteration history ===")
iter_rows = []
for h in v3_history:
    assets = h.get("assets")
    if assets is None:
        tagline = "(parse failed)"
    else:
        tagline = (assets.get("tagline") or "")[:120]
    avg = h.get("average")
    avg_str = f"{avg:.2f}" if isinstance(avg, (int, float)) else "n/a"
    iter_rows.append(
        {
            "iter": h["iter"],
            "tagline": tagline,
            "average": avg_str,
            "passed": h["passed"],
        }
    )
display(pd.DataFrame(iter_rows))

print("\n=== Overall averages: V1 / V2 / V3 ===")
print(f"V1 average: {result_v1['average']:.2f}")
print(f"V2 average: {result_v2['average']:.2f}")
print(f"V3 average: {result_v3['average']:.2f}")

rows = []
for dim in DIMENSIONS:
    s1 = result_v1["scores"].get(dim, {}).get("score", None)
    s2 = result_v2["scores"].get(dim, {}).get("score", None)
    s3 = result_v3["scores"].get(dim, {}).get("score", None)
    candidates = {"V1": s1, "V2": s2, "V3": s3}
    valid = {k: v for k, v in candidates.items() if isinstance(v, (int, float))}
    best = max(valid, key=valid.get) if valid else None
    rows.append(
        {
            "dimension": dim,
            "V1": s1,
            "V2": s2,
            "V3": s3,
            "best": best,
        }
    )

comparison_df_v123 = pd.DataFrame(rows)
print("\n=== Per-dimension comparison (V1 / V2 / V3) ===")
display(comparison_df_v123)


[v3 iter 1] tagline='Raindrops mask hidden truths' average=4.40

  average: 4.20 / 5.00
  narrative_fidelity           █████  5/5
    → The content strictly adheres to the core premise and thematic core without violating any constraints.
  genre_alignment              █████  5/5
    → The language and tone are fully consistent with the psychological thriller genre.
  visual_specificity           ███░░  3/5
    → The description mentions the atmosphere and mood but lacks detailed visual elements like lighting or color palette.
  creative_specificity         ███░░  3/5
    → The tagline is competent and on-brief without offering a distinct and memorable phrase.
  output_format_validity       █████  5/5
    → All fields are present and within the specified word limits.

=== V3: Final output ===
{'overview': 'In the midst of relentless rains, a small clinic on the edge of '
             'a rural village struggles to survive, isolating its patients and '
             'staff. As tensions ris

,iter,tagline,average,passed
0,1,Raindrops mask hidden truths,4.40,True



=== Overall averages: V1 / V2 / V3 ===
V1 average: 4.60
V2 average: 4.40
V3 average: 4.20

=== Per-dimension comparison (V1 / V2 / V3) ===


,dimension,V1,V2,V3,best
0,narrative_fidelity,5,5,5,V1
1,genre_alignment,5,5,5,V1
2,visual_specificity,3,3,3,V1
3,creative_specificity,5,4,3,V1
4,output_format_validity,5,5,5,V1


In [38]:
# Cell 8-F: Full score summary + generated content for team sharing

variants = {
    "V1 (zero-shot)": (result_v1, v1),
    "V2 (RAG+visual)": (result_v2, v2),
    "V3 (agentic)":    (result_v3, v3),
}

# ── Score summary table ──────────────────────────────────────────────
rows = []
for variant_name, (result, _) in variants.items():
    row = {"variant": variant_name, "average": result["average"]}
    for dim in DIMENSIONS:
        row[dim] = result["scores"].get(dim, {}).get("score", None)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("variant")
print("=" * 70)
print("SCORE SUMMARY")
print("=" * 70)
display(summary_df)

# ── Generated content + reasons side by side ─────────────────────────
print("\n" + "=" * 70)
print("GENERATED CONTENT + JUDGE REASONING")
print("=" * 70)

for variant_name, (result, output) in variants.items():
    avg = result["average"]
    print(f"\n{'─' * 70}")
    print(f"  {variant_name}   (average: {avg:.2f} / 5.00)")
    print(f"{'─' * 70}")
    print(f"\n  [TAGLINE]\n  {output.get('tagline', '—')}")
    print(f"\n  [OVERVIEW]\n  {output.get('overview', '—')}")
    print(f"\n  [POSTER ART DIRECTION]\n  {output.get('poster_art_direction', '—')}")
    print(f"\n  [JUDGE SCORES]")
    for dim in DIMENSIONS:
        entry  = result["scores"].get(dim, {})
        score  = entry.get("score", "?")
        reason = entry.get("reason", "")
        bar    = "█" * score + "░" * (5 - score) if isinstance(score, int) else ""
        print(f"  {dim:<28} {bar}  {score}/5")
        print(f"    → {reason}")

# ── Key observations (auto-generated from results) ────────────────────
print(f"\n{'=' * 70}")
print("KEY OBSERVATIONS")
print("=" * 70)

avgs = {k: v[0]["average"] for k, v in variants.items()}
for name, avg in avgs.items():
    print(f"  {name}: {avg:.2f}")

# Best and worst variant
best_variant  = max(avgs, key=avgs.get)
worst_variant = min(avgs, key=avgs.get)
print(f"\n  Best : {best_variant} ({avgs[best_variant]:.2f})")
print(f"  Worst: {worst_variant} ({avgs[worst_variant]:.2f})")

# Dimensions where variants differ
print(f"\n  Dimensions with score differences:")
any_diff = False
for dim in DIMENSIONS:
    scores = {k: v[0]["scores"].get(dim, {}).get("score", 0) for k, v in variants.items()}
    if len(set(scores.values())) > 1:
        any_diff = True
        best_v  = max(scores, key=scores.get)
        worst_v = min(scores, key=scores.get)
        print(f"  ✦ {dim:<28} {scores}")
        print(f"      best={best_v}({scores[best_v]}), worst={worst_v}({scores[worst_v]})")
        # Print judge reason for worst
        reason = variants[worst_v][0]["scores"].get(dim, {}).get("reason", "")
        if reason:
            print(f"      judge on {worst_v}: '{reason}'")
if not any_diff:
    print("  No dimension-level differences found across variants.")

# Dimensions stuck at same score across all variants
print(f"\n  Dimensions with identical scores across all variants (potential ceiling):")
any_flat = False
for dim in DIMENSIONS:
    scores = {k: v[0]["scores"].get(dim, {}).get("score", 0) for k, v in variants.items()}
    if len(set(scores.values())) == 1:
        any_flat = True
        val = list(scores.values())[0]
        flag = "⚠ ceiling" if val >= 4 else "⚠ floor"
        print(f"  {flag} {dim:<28} all={val}/5")
if not any_flat:
    print("  None.")

# V3 iteration check
v3_result = variants["V3 (agentic)"][0]
if "history" in dir():
    n_iters = len([h for h in v3_history if h.get("assets") is not None])
    passed_at = next((h["iter"] for h in v3_history if h.get("passed")), None)
    print(f"\n  V3 agentic loop: {n_iters} iteration(s), early-stopped at iter {passed_at}")
else:
    print(f"\n  V3 average: {v3_result['average']:.2f} (iteration history not available in this scope)")

v1_idea1, v2_idea1, v3_idea1 = v1, v2, v3
result_v1_idea1, result_v2_idea1, result_v3_idea1 = result_v1, result_v2, result_v3

SCORE SUMMARY


,average,narrative_fidelity,genre_alignment,visual_specificity,creative_specificity,output_format_validity
variant,,,,,,
V1 (zero-shot),4.6,5,5,3,5,5
V2 (RAG+visual),4.4,5,5,3,4,5
V3 (agentic),4.2,5,5,3,3,5



GENERATED CONTENT + JUDGE REASONING

──────────────────────────────────────────────────────────────────────
  V1 (zero-shot)   (average: 4.60 / 5.00)
──────────────────────────────────────────────────────────────────────

  [TAGLINE]
  Rain keeps them out. Fear keeps them in.

  [OVERVIEW]
  In a remote, rain-soaked village, a community gathers at the lone clinic, their lives intertwined by necessity and circumstance. As the floodwaters rise, tensions simmer beneath the surface, threatening to boil over in unexpected ways.

  [POSTER ART DIRECTION]
  A hauntingly beautiful image of the village shrouded in rain, the clinic standing defiantly amidst the chaos, the faces of the villagers peering through windows reflecting both fear and determination.

  [JUDGE SCORES]
  narrative_fidelity           █████  5/5
    → The assets strictly adhere to the core premise and thematic core without violating constraints.
  genre_alignment              █████  5/5
    → The language and mood perfectly

In [ ]:
# Cell 8-G: Run V1 / V2 / V3 experiments for Idea 2 and Idea 3

mmovie_input_idea2 = {
    "core_premise": (
        "Twin brothers separated at birth are reunited at age 35 — "
        "one raised in wealth and privilege in Tokyo, "
        "the other in poverty in rural Tohoku. "
        "Forced together by their biological mother's death and a contested inheritance, "
        "they spend two weeks in the same house for the first time."
    ),
    "thematic_core": (
        "Nature versus nurture made flesh: two men who share everything biological "
        "and nothing experiential, trying to recognize themselves in each other."
    ),
    "negative_constraints": (
        "Do NOT resolve the class tension with empathy or reconciliation. "
        "The divide should feel real and irreducible, not a lesson to be learned. "
        "Do NOT make either twin the clear hero or villain."
    ),
}

movie_input_idea3 = {
    "core_premise": (
        "An ordinary 28-year-old data analyst wakes up one morning "
        "able to predict events exactly 47 seconds before they happen. "
        "Within 72 hours, three governments and a private intelligence firm "
        "are hunting her. She finds others with abilities — "
        "but they are not all on her side."
    ),
    "thematic_core": (
        "The burden of seeing what comes next: "
        "knowing the future does not mean you can change it, "
        "and the people who want to control you know that too."
    ),
    "negative_constraints": (
        "Do NOT give the protagonist combat training or physical superiority. "
        "Her power must feel like a curse as much as an advantage. "
        "Do NOT introduce a single mastermind villain — "
        "the threat should be systemic and faceless."
    ),
}


def _trend_label(a1, a2, a3):
    """V1<V2<V3 when strictly increasing; otherwise ascending score order with '=' for ties."""
    if a1 < a2 < a3:
        return "V1<V2<V3"
    pairs = sorted([("V1", a1), ("V2", a2), ("V3", a3)], key=lambda x: (x[1], x[0]))
    s = pairs[0][0]
    for i in range(1, 3):
        sep = "=" if pairs[i][1] == pairs[i - 1][1] else "<"
        s += sep + pairs[i][0]
    return s


def _print_bundle_like_8f(idea_label, v3_history_local):
    """Score summary table, generated content + judge reasoning, key observations (same style as Cell 8-F)."""
    variants = {
        "V1 (zero-shot)": (result_v1, v1),
        "V2 (RAG+visual)": (result_v2, v2),
        "V3 (agentic)": (result_v3, v3),
    }

    print("\n" + "#" * 70)
    print(f"# {idea_label}")
    print("#" * 70)

    rows = []
    for variant_name, (result, _) in variants.items():
        row = {"variant": variant_name, "average": result["average"]}
        for dim in DIMENSIONS:
            row[dim] = result["scores"].get(dim, {}).get("score", None)
        rows.append(row)

    summary_df = pd.DataFrame(rows).set_index("variant")
    print("=" * 70)
    print("SCORE SUMMARY")
    print("=" * 70)
    display(summary_df)

    print("\n" + "=" * 70)
    print("GENERATED CONTENT + JUDGE REASONING")
    print("=" * 70)

    for variant_name, (result, output) in variants.items():
        avg = result["average"]
        print(f"\n{'─' * 70}")
        print(f"  {variant_name}   (average: {avg:.2f} / 5.00)")
        print(f"{'─' * 70}")
        print(f"\n  [TAGLINE]\n  {output.get('tagline', '—')}")
        print(f"\n  [OVERVIEW]\n  {output.get('overview', '—')}")
        print(f"\n  [POSTER ART DIRECTION]\n  {output.get('poster_art_direction', '—')}")
        print(f"\n  [JUDGE SCORES]")
        for dim in DIMENSIONS:
            entry = result["scores"].get(dim, {})
            score = entry.get("score", "?")
            reason = entry.get("reason", "")
            bar = "█" * score + "░" * (5 - score) if isinstance(score, int) else ""
            print(f"  {dim:<28} {bar}  {score}/5")
            print(f"    → {reason}")

    print(f"\n{'=' * 70}")
    print("KEY OBSERVATIONS")
    print("=" * 70)

    avgs = {k: v[0]["average"] for k, v in variants.items()}
    for name, avg in avgs.items():
        print(f"  {name}: {avg:.2f}")

    best_variant = max(avgs, key=avgs.get)
    worst_variant = min(avgs, key=avgs.get)
    print(f"\n  Best : {best_variant} ({avgs[best_variant]:.2f})")
    print(f"  Worst: {worst_variant} ({avgs[worst_variant]:.2f})")

    print(f"\n  Dimensions with score differences:")
    any_diff = False
    for dim in DIMENSIONS:
        scores = {k: v[0]["scores"].get(dim, {}).get("score", 0) for k, v in variants.items()}
        if len(set(scores.values())) > 1:
            any_diff = True
            best_v = max(scores, key=scores.get)
            worst_v = min(scores, key=scores.get)
            print(f"  ✦ {dim:<28} {scores}")
            print(f"      best={best_v}({scores[best_v]}), worst={worst_v}({scores[worst_v]})")
            reason = variants[worst_v][0]["scores"].get(dim, {}).get("reason", "")
            if reason:
                print(f"      judge on {worst_v}: '{reason}'")
    if not any_diff:
        print("  No dimension-level differences found across variants.")

    print(f"\n  Dimensions with identical scores across all variants (potential ceiling):")
    any_flat = False
    for dim in DIMENSIONS:
        scores = {k: v[0]["scores"].get(dim, {}).get("score", 0) for k, v in variants.items()}
        if len(set(scores.values())) == 1:
            any_flat = True
            val = list(scores.values())[0]
            flag = "⚠ ceiling" if val >= 4 else "⚠ floor"
            print(f"  {flag} {dim:<28} all={val}/5")
    if not any_flat:
        print("  None.")

    v3_result = variants["V3 (agentic)"][0]
    if v3_history_local is not None:
        n_iters = len([h for h in v3_history_local if h.get("assets") is not None])
        passed_at = next((h["iter"] for h in v3_history_local if h.get("passed")), None)
        print(f"\n  V3 agentic loop: {n_iters} iteration(s), early-stopped at iter {passed_at}")
    else:
        print(f"\n  V3 average: {v3_result['average']:.2f} (iteration history not available in this scope)")


# Cross-idea table: Idea 1 uses results from Cells 8-C / 8-E / 8-F (run those first).
cross_rows = [
    {
        "variant": "Idea 1",
        "V1_avg": result_v1["average"],
        "V2_avg": result_v2["average"],
        "V3_avg": result_v3["average"],
        "trend": _trend_label(
            result_v1["average"], result_v2["average"], result_v3["average"]
        ),
    }
]

for idea_label, movie_input in [
    ("Idea 2", movie_input_idea2),
    ("Idea 3", movie_input_idea3),
]:
    v1, v2, topk, refs = run_v1_v2(
        movie_input, k=5, genre_filter=None, temperature=0.6
    )
    judge_client = OpenAI()
    v3, v3_history = run_v3(
        movie_input,
        k=5,
        genre_filter=None,
        temperature=0.6,
        judge_client=judge_client,
    )
    if idea_label == "Idea 2":
        v1_idea2, v2_idea2, v3_idea2 = v1, v2, v3
    elif idea_label == "Idea 3":
        v1_idea3, v2_idea3, v3_idea3 = v1, v2, v3
    result_v1 = llm_as_judge(movie_input, v1, client=judge_client, verbose=False)
    result_v2 = llm_as_judge(movie_input, v2, client=judge_client, verbose=False)
    result_v3 = llm_as_judge(movie_input, v3, client=judge_client, verbose=False)
    if idea_label == "Idea 2":
        result_v1_idea2 = result_v1
        result_v2_idea2 = result_v2
        result_v3_idea2 = result_v3
    elif idea_label == "Idea 3":
        result_v1_idea3 = result_v1
        result_v2_idea3 = result_v2
        result_v3_idea3 = result_v3

    _print_bundle_like_8f(idea_label, v3_history)

    cross_rows.append(
        {
            "variant": idea_label,
            "V1_avg": result_v1["average"],
            "V2_avg": result_v2["average"],
            "V3_avg": result_v3["average"],
            "trend": _trend_label(
                result_v1["average"], result_v2["average"], result_v3["average"]
            ),
        }
    )

print("\n" + "=" * 70)
print("CROSS-IDEA SUMMARY (average scores)")
print("=" * 70)
cross_df = pd.DataFrame(cross_rows)[
    ["variant", "V1_avg", "V2_avg", "V3_avg", "trend"]
]
display(cross_df)


[v3 iter 1] tagline='Two sisters, one home, unspoken truths' average=4.00

######################################################################
# Idea 2
######################################################################
SCORE SUMMARY


,average,narrative_fidelity,genre_alignment,visual_specificity,creative_specificity,output_format_validity
variant,,,,,,
V1 (zero-shot),4.6,5,5,5,3,5
V2 (RAG+visual),4.6,5,5,5,3,5
V3 (agentic),4.4,5,5,3,4,5



GENERATED CONTENT + JUDGE REASONING

──────────────────────────────────────────────────────────────────────
  V1 (zero-shot)   (average: 4.60 / 5.00)
──────────────────────────────────────────────────────────────────────

  [TAGLINE]
  Two Weeks. Two Sisters. One Legacy.

  [OVERVIEW]
  In the tranquil countryside of rural Japan, an estranged pair of sisters confront unspoken grief and simmering resentment when they're forced to live under one roof for two weeks, settling their late mother's affairs.

  [POSTER ART DIRECTION]
  A serene landscape image of a Japanese garden, where two sisters stand back-to-back, gazing at an empty house in the distance. Their silhouettes cast long shadows on the ground, symbolizing the deep divide between them.

  [JUDGE SCORES]
  narrative_fidelity           █████  5/5
    → The output strictly adheres to the premise and thematic core without violating any constraints.
  genre_alignment              █████  5/5
    → Language and emotional tone align p

,average,narrative_fidelity,genre_alignment,visual_specificity,creative_specificity,output_format_validity
variant,,,,,,
V1 (zero-shot),4.2,5,5,3,3,5
V2 (RAG+visual),4.6,5,5,5,3,5
V3 (agentic),4.2,5,4,4,3,5



GENERATED CONTENT + JUDGE REASONING

──────────────────────────────────────────────────────────────────────
  V1 (zero-shot)   (average: 4.20 / 5.00)
──────────────────────────────────────────────────────────────────────

  [TAGLINE]
  Expertise has its price.

  [OVERVIEW]
  When the city's lifeblood is targeted by an enigmatic foe, a seasoned bomb disposal expert must once more don his suit and save the day – but at what cost?

  [POSTER ART DIRECTION]
  A tense, nighttime scene depicts our hero poised against a ticking time bomb, the city skyline ominously reflected in still waters behind him.

  [JUDGE SCORES]
  narrative_fidelity           █████  5/5
    → The assets maintain adherence to the core premise and thematic core with no constraint violations.
  genre_alignment              █████  5/5
    → The language and tone align perfectly with the action-thriller genre.
  visual_specificity           ███░░  3/5
    → The description provides a mood but lacks details such as lighti

,variant,V1_avg,V2_avg,V3_avg,trend
0,Idea 1,4.6,4.4,4.2,V3<V2<V1
1,Idea 2,4.6,4.6,4.4,V3<V1=V2
2,Idea 3,4.2,4.6,4.2,V1=V3<V2


In [40]:
# Cell 8-H: Poster image generation using Nano Banana (gemini-3.1-flash-image-preview)
# Uses generate_content() with response_modalities=['IMAGE']
# Cost: $0.039/image × 9 = ~$0.35 total

import subprocess
import shutil
import base64
from google import genai as new_genai
from google.genai import types as new_types

POSTER_DIR = os.path.join(REPO_ROOT, "poster_images")

# Force regenerate: clear existing images
if os.path.exists(POSTER_DIR):
    shutil.rmtree(POSTER_DIR)
os.makedirs(POSTER_DIR)
print("✅ Cleared existing poster images — regenerating all 9")

nano_client = new_genai.Client(api_key=GEMINI_API_KEY)

poster_inputs = {
    "idea1_v1": v1_idea1.get("poster_art_direction", ""),
    "idea1_v2": v2_idea1.get("poster_art_direction", ""),
    "idea1_v3": v3_idea1.get("poster_art_direction", ""),
    "idea2_v1": v1_idea2.get("poster_art_direction", ""),
    "idea2_v2": v2_idea2.get("poster_art_direction", ""),
    "idea2_v3": v3_idea2.get("poster_art_direction", ""),
    "idea3_v1": v1_idea3.get("poster_art_direction", ""),
    "idea3_v2": v2_idea3.get("poster_art_direction", ""),
    "idea3_v3": v3_idea3.get("poster_art_direction", ""),
}

_stats = {"generated": 0, "skipped": 0, "failed": 0}

def generate_poster(label, art_direction):
    out_path = os.path.join(POSTER_DIR, f"{label}.png")

    if not (art_direction or "").strip():
        print(f"[{label}] Skipped: no art direction")
        _stats["skipped"] += 1
        return ""

    prompt = (
        "A professional cinematic movie poster. "
        f"{art_direction} "
        "Film poster style, high contrast, dramatic lighting. "
        "No text, no title, no credits — image only. "
        "Portrait orientation (9:16 aspect ratio)."
    )

    try:
        response = nano_client.models.generate_content(
            model="gemini-3.1-flash-image-preview",
            contents=prompt,
            config=new_types.GenerateContentConfig(
                response_modalities=["IMAGE"],
                image_config=new_types.ImageConfig(
                    aspect_ratio="9:16",
                    image_size="1K",
                ),
            ),
        )
        # Extract image from response parts
        for part in response.parts:
            if part.inline_data is not None:
                image_bytes = part.inline_data.data
                with open(out_path, "wb") as f:
                    f.write(image_bytes)
                print(f"[{label}] ✅ Saved: {out_path}")
                _stats["generated"] += 1
                return out_path

        print(f"[{label}] ❌ No image in response")
        _stats["failed"] += 1
        return ""

    except Exception as e:
        print(f"[{label}] ❌ Failed: {e}")
        _stats["failed"] += 1
        return ""

saved_paths = {}
for lbl, art in poster_inputs.items():
    saved_paths[lbl] = generate_poster(lbl, art)

print(f"\nSummary: {_stats['generated']} generated / "
      f"{_stats['skipped']} skipped / {_stats['failed']} failed")

if _stats["generated"] > 0:
    subprocess.run(["git", "-C", REPO_ROOT, "add", "poster_images/"], check=False)
    commit_proc = subprocess.run(
        ["git", "-C", REPO_ROOT, "commit", "-m",
         f"update: Nano Banana poster images ({_stats['generated']} generated)"],
        capture_output=True, text=True
    )
    print("git commit:", commit_proc.stdout.strip() or commit_proc.stderr.strip())
    push_proc = subprocess.run(
        ["git", "-C", REPO_ROOT, "push"],
        capture_output=True, text=True
    )
    print("git push:", push_proc.stdout.strip() or push_proc.stderr.strip() or "ok")
else:
    print("⚠ No images generated — skipping git commit")

✅ Cleared existing poster images — regenerating all 9
[idea1_v1] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea1_v1.png
[idea1_v2] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea1_v2.png
[idea1_v3] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea1_v3.png
[idea2_v1] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea2_v1.png
[idea2_v2] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea2_v2.png
[idea2_v3] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea2_v3.png
[idea3_v1] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea3_v1.png
[idea3_v2] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea3_v2.png
[idea3_v3] ✅ Saved: ryuichi-github-info290-2026s-group5/poster_images/idea3_v3.png

Summary: 9 generated / 0 skipped / 0 failed
git commit: [caption-batch 14a7da8] update: Nano Banana poster images (9 generated)
 9 files changed, 0 insertions(+), 0 deletions(-)
 rewrite poster_imag

In [41]:
# Cell 8-I: LLM Generator comparison — V1 vs V2 across Mistral-7B, Gemini 2.5 Flash, GPT-4o-mini
# Run on all three Ideas (Idea 1 / 2 / 3) for statistical credibility.

import time
from openai import OpenAI


# --- 1) Gemini 2.5 Flash generator ---
def generate_text_gemini(prompt: str) -> str:
    try:
        resp = gemini.generate_content(prompt)
        return resp.text or ""
    except Exception as e:
        print(f"[gemini] warning: {e}")
        return ""


def run_v1_v2_gemini(movie_input, k=5, genre_filter=None):
    # V1
    p1 = build_prompt(movie_input, refs="(none)")
    t1 = generate_text_gemini(p1)
    j1 = extract_best_json(t1)

    # V2
    query = movie_input["core_premise"]
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)
    p2 = build_prompt(movie_input, refs=refs)
    t2 = generate_text_gemini(p2)
    j2 = extract_best_json(t2)

    return j1, j2, topk, refs


# --- 2) GPT-4o-mini generator ---
def generate_text_openai(prompt: str, model: str = "gpt-4o-mini") -> str:
    try:
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.6,
            max_tokens=400,
        )
        return resp.choices[0].message.content or ""
    except Exception as e:
        print(f"[openai:{model}] warning: {e}")
        return ""


def run_v1_v2_openai(movie_input, k=5, genre_filter=None):
    # V1
    p1 = build_prompt(movie_input, refs="(none)")
    t1 = generate_text_openai(p1)
    j1 = extract_best_json(t1)

    # V2
    query = movie_input["core_premise"]
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)
    p2 = build_prompt(movie_input, refs=refs)
    t2 = generate_text_openai(p2)
    j2 = extract_best_json(t2)

    return j1, j2, topk, refs


# --- 3) Run all three generators on Ideas 1, 2, and 3 ---
idea_configs_gen = [
    {
        "id": 1,
        "movie_input": movie_input,
        "genre_filter": ["thriller", "horror", "mystery"],
        "mistral_v1": v1_idea1,
        "mistral_v2": v2_idea1,
    },
    {
        "id": 2,
        "movie_input": movie_input_idea2,
        "genre_filter": None,
        "mistral_v1": v1_idea2,
        "mistral_v2": v2_idea2,
    },
    {
        "id": 3,
        "movie_input": movie_input_idea3,
        "genre_filter": None,
        "mistral_v1": v1_idea3,
        "mistral_v2": v2_idea3,
    },
]

gen_results = {}


# --- 4) Evaluate all outputs with llm_as_judge(verbose=False) ---
judge_client = OpenAI()

for cfg in idea_configs_gen:
    idea_id = int(cfg["id"])
    movie_input_cfg = cfg["movie_input"]
    gf = cfg["genre_filter"]

    gen_results[idea_id] = {
        "mistral": {
            "v1": cfg["mistral_v1"],
            "v2": cfg["mistral_v2"],
        },
        "gemini": {},
        "openai": {},
    }

    # Gemini
    g_v1, g_v2, _g_topk, _g_refs = run_v1_v2_gemini(movie_input_cfg, k=5, genre_filter=gf)
    gen_results[idea_id]["gemini"]["v1"] = g_v1
    gen_results[idea_id]["gemini"]["v2"] = g_v2

    # GPT-4o-mini
    o_v1, o_v2, _o_topk, _o_refs = run_v1_v2_openai(movie_input_cfg, k=5, genre_filter=gf)
    gen_results[idea_id]["openai"]["v1"] = o_v1
    gen_results[idea_id]["openai"]["v2"] = o_v2

    # Judge (V1/V2 for each generator)
    for gen_name in ["mistral", "gemini", "openai"]:
        for var in ["v1", "v2"]:
            out = gen_results[idea_id][gen_name][var]
            judge_out = llm_as_judge(movie_input_cfg, out, client=judge_client, verbose=False)
            gen_results[idea_id][gen_name][f"result_{var}"] = judge_out
            time.sleep(0.5)


# --- 5) Print generated content for each idea × generator (Cell 8-F format) ---
def _print_variant_block(header: str, output: dict, result: dict):
    avg = result.get("average", 0.0)
    print(f"\n{'─' * 70}")
    print(f"  {header}   (average: {avg:.2f} / 5.00)")
    print(f"{'─' * 70}")
    print(f"\n  [TAGLINE]\n  {output.get('tagline', '—')}")
    print(f"\n  [OVERVIEW]\n  {output.get('overview', '—')}")
    print(f"\n  [POSTER ART DIRECTION]\n  {output.get('poster_art_direction', '—')}")
    print(f"\n  [JUDGE SCORES]")
    for dim in DIMENSIONS:
        entry = result.get("scores", {}).get(dim, {})
        score = entry.get("score", "?")
        reason = entry.get("reason", "")
        bar = "█" * score + "░" * (5 - score) if isinstance(score, int) else ""
        print(f"  {dim:<28} {bar}  {score}/5")
        print(f"    → {reason}")


gen_display_names = {
    "mistral": "Mistral-7B",
    "gemini": "Gemini 2.5 Flash",
    "openai": "GPT-4o-mini",
}

for cfg in idea_configs_gen:
    idea_id = int(cfg["id"])
    print("\n" + "=" * 70)
    print(f"IDEA {idea_id} — GENERATOR COMPARISON (V1 vs V2)")
    print("=" * 70)

    for gen_name in ["mistral", "gemini", "openai"]:
        g = gen_results[idea_id][gen_name]
        _print_variant_block(
            f"Idea {idea_id} — {gen_display_names[gen_name]} — V1",
            g["v1"],
            g["result_v1"],
        )
        _print_variant_block(
            f"Idea {idea_id} — {gen_display_names[gen_name]} — V2",
            g["v2"],
            g["result_v2"],
        )


# --- 6) Print per-idea comparison table ---
for cfg in idea_configs_gen:
    idea_id = int(cfg["id"])
    rows = []
    for gen_name in ["mistral", "gemini", "openai"]:
        g = gen_results[idea_id][gen_name]
        v1_avg = float(g["result_v1"].get("average", 0.0))
        v2_avg = float(g["result_v2"].get("average", 0.0))
        rows.append(
            {
                "generator": gen_display_names[gen_name],
                "V1_avg": round(v1_avg, 2),
                "V2_avg": round(v2_avg, 2),
                "delta": round(v2_avg - v1_avg, 2),
            }
        )

    idea_df = pd.DataFrame(rows)[["generator", "V1_avg", "V2_avg", "delta"]]
    print("\n" + "=" * 70)
    print(f"IDEA {idea_id} — PER-GENERATOR COMPARISON")
    print("=" * 70)
    display(idea_df)


# --- 7) Print cross-idea summary table ---
rows = []
for gen_name in ["mistral", "gemini", "openai"]:
    deltas = []
    row = {"generator": gen_display_names[gen_name]}
    for idea_id in [1, 2, 3]:
        g = gen_results[idea_id][gen_name]
        v1_avg = float(g["result_v1"].get("average", 0.0))
        v2_avg = float(g["result_v2"].get("average", 0.0))
        row[f"idea{idea_id}_V1"] = round(v1_avg, 2)
        row[f"idea{idea_id}_V2"] = round(v2_avg, 2)
        deltas.append(v2_avg - v1_avg)
    row["avg_delta"] = round(sum(deltas) / len(deltas), 2)
    rows.append(row)

cross_df = pd.DataFrame(rows)[
    [
        "generator",
        "idea1_V1",
        "idea1_V2",
        "idea2_V1",
        "idea2_V2",
        "idea3_V1",
        "idea3_V2",
        "avg_delta",
    ]
]

print("\n" + "=" * 70)
print("CROSS-IDEA SUMMARY — Which generator benefits most from RAG (V2 > V1)?")
print("=" * 70)
display(cross_df)



IDEA 1 — GENERATOR COMPARISON (V1 vs V2)

──────────────────────────────────────────────────────────────────────
  Idea 1 — Mistral-7B — V1   (average: 2.60 / 5.00)
──────────────────────────────────────────────────────────────────────

  [TAGLINE]
  Rain keeps them out. Fear keeps them in.

  [OVERVIEW]
  In a remote, rain-soaked village, a community gathers at the lone clinic, their lives intertwined by necessity and circumstance. As the floodwaters rise, tensions simmer beneath the surface, threatening to boil over in unexpected ways.

  [POSTER ART DIRECTION]
  A hauntingly beautiful image of the village shrouded in rain, the clinic standing defiantly amidst the chaos, the faces of the villagers peering through windows reflecting both fear and determination.

  [JUDGE SCORES]
  narrative_fidelity           █░░░░  1/5
    → The generated content does not match the core premise and introduces unrelated characters and settings.
  genre_alignment              █░░░░  1/5
    → The lang

,generator,V1_avg,V2_avg,delta
0,Mistral-7B,2.6,2.2,-0.4
1,Gemini 2.5 Flash,4.6,5.0,0.4
2,GPT-4o-mini,4.2,4.6,0.4



IDEA 2 — PER-GENERATOR COMPARISON


,generator,V1_avg,V2_avg,delta
0,Mistral-7B,4.6,4.2,-0.4
1,Gemini 2.5 Flash,5.0,5.0,0.0
2,GPT-4o-mini,4.4,4.8,0.4



IDEA 3 — PER-GENERATOR COMPARISON


,generator,V1_avg,V2_avg,delta
0,Mistral-7B,4.0,4.6,0.6
1,Gemini 2.5 Flash,4.8,4.0,-0.8
2,GPT-4o-mini,4.2,4.6,0.4



CROSS-IDEA SUMMARY — Which generator benefits most from RAG (V2 > V1)?


,generator,idea1_V1,idea1_V2,idea2_V1,idea2_V2,idea3_V1,idea3_V2,avg_delta
0,Mistral-7B,2.6,2.2,4.6,4.2,4.0,4.6,-0.07
1,Gemini 2.5 Flash,4.6,5.0,5.0,5.0,4.8,4.0,-0.13
2,GPT-4o-mini,4.2,4.6,4.4,4.8,4.2,4.6,0.40


In [ ]:
# Cell 8-J-0: Nano Banana connectivity check
# Verify both preview image models are reachable with GEMINI_API_KEY before the full experiment.

from google import genai as new_genai
from google.genai import types as new_types
from IPython.display import display, Image as IPImage
import io

nano_client = new_genai.Client(api_key=GEMINI_API_KEY)


def test_nano_banana(model_name: str) -> bool:
    """Minimal image generation probe; saves to /tmp and displays inline."""
    prompt = (
        "A single red apple on a white surface.\n"
        "Photorealistic, studio lighting. No text."
    )
    safe = model_name.replace("/", "_").replace(":", "_")
    out_path = f"/tmp/test_{safe}.png"
    try:
        # Some SDK versions reject certain image_size strings; fall back if needed.
        try:
            img_cfg = new_types.ImageConfig(aspect_ratio="1:1", image_size="512")
        except Exception:
            img_cfg = new_types.ImageConfig(aspect_ratio="1:1")

        resp = nano_client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=new_types.GenerateContentConfig(
                response_modalities=["IMAGE"],
                image_config=img_cfg,
            ),
        )

        data = None
        for cand in getattr(resp, "candidates", []) or []:
            content = getattr(cand, "content", None)
            if not content:
                continue
            for part in getattr(content, "parts", []) or []:
                inline = getattr(part, "inline_data", None)
                if inline is not None and getattr(inline, "data", None):
                    data = inline.data
                    break
            if data:
                break

        if not data:
            print(f"❌ {model_name} FAILED: no inline image data in response")
            return False

        raw = data
        if isinstance(raw, str):
            import base64

            raw = base64.b64decode(raw)

        with open(out_path, "wb") as f:
            f.write(raw)
        display(IPImage(filename=out_path))
        print(f"✅ {model_name} OK")
        return True
    except Exception as e:
        print(f"❌ {model_name} FAILED: {e}")
        return False


flash_ok = test_nano_banana("gemini-3.1-flash-image-preview")
pro_ok = test_nano_banana("gemini-3-pro-image-preview")

print(f"Flash: {'✅' if flash_ok else '❌'}")
print(f"Pro:   {'✅' if pro_ok else '❌'}")

NANO_MODEL = "gemini-3-pro-image-preview" if pro_ok else "gemini-3.1-flash-image-preview"
print(f"Using model for experiments: {NANO_MODEL}")


In [ ]:
# Cell 8-J-1: Prompt approach × model comparison experiment
# 3 approaches (A/B/C) × up to 2 image models × 3 ideas × 3 variants (v1/v2/v3).
# Total images: flash only  → 3 × 1 × 3 × 3 = 27
#               flash + pro → 3 × 2 × 3 × 3 = 54

import os
import re
import shutil
import time
from openai import OpenAI
from google.genai import types as new_types

COMPARE_DIR = os.path.join(REPO_ROOT, "poster_compare")
FORCE_REGENERATE = False  # Set True to clear and regenerate all

if FORCE_REGENERATE and os.path.exists(COMPARE_DIR):
    shutil.rmtree(COMPARE_DIR)
    print("✅ Cleared COMPARE_DIR")
os.makedirs(COMPARE_DIR, exist_ok=True)

existing = len([f for f in os.listdir(COMPARE_DIR) if f.endswith(".png")])
print(f"COMPARE_DIR: {existing} existing images. FORCE_REGENERATE={FORCE_REGENERATE}")

POSTER_SUFFIX = "\nPhotorealistic. No text. No title. No credits. Portrait orientation."

idea_configs = [
    {
        "id": 1,
        "label": "idea1",
        "genre": "Psychological Thriller",
        "movie_input": movie_input,
        "variants": {
            "v1": v1_idea1.get("poster_art_direction", ""),
            "v2": v2_idea1.get("poster_art_direction", ""),
            "v3": v3_idea1.get("poster_art_direction", ""),
        },
    },
    {
        "id": 2,
        "label": "idea2",
        "genre": "Drama / Family Loss",
        "movie_input": movie_input_idea2,
        "variants": {
            "v1": v1_idea2.get("poster_art_direction", ""),
            "v2": v2_idea2.get("poster_art_direction", ""),
            "v3": v3_idea2.get("poster_art_direction", ""),
        },
    },
    {
        "id": 3,
        "label": "idea3",
        "genre": "Action Thriller",
        "movie_input": movie_input_idea3,
        "variants": {
            "v1": v1_idea3.get("poster_art_direction", ""),
            "v2": v2_idea3.get("poster_art_direction", ""),
            "v3": v3_idea3.get("poster_art_direction", ""),
        },
    },
]

VARIANT_KEYS = ["v1", "v2", "v3"]

# Models to use — pro_ok is set in Cell 8-J-0
MODELS = {"flash": "gemini-3.1-flash-image-preview"}
if pro_ok:
    MODELS["pro"] = "gemini-3-pro-image-preview"
print(f"Models: {list(MODELS.keys())}")


# --- Approach A: short art_direction + POSTER_SUFFIX ---
art_a = {}
for cfg in idea_configs:
    art_a[cfg["label"]] = {}
    for vk in VARIANT_KEYS:
        art_a[cfg["label"]][vk] = cfg["variants"][vk].strip() + POSTER_SUFFIX


# --- Approach B: Mistral-7B extended (~150 words) ---
def generate_extended_art_direction(movie_input: dict, genre: str, seed: str) -> str:
    premise = movie_input.get("core_premise", "")
    theme = movie_input.get("thematic_core", "")
    negs = movie_input.get("negative_constraints", "")
    prompt = f"""You are a senior movie key-art director.

Write ONE movie poster IMAGE DESCRIPTION in plain English (NOT JSON), 120 to 150 words.
Cover ALL of:
(1) Subject and their position/pose in frame
(2) Lighting: direction, color temperature, shadow quality
(3) Color palette: exactly 3 specific colors
(4) Camera angle and shot type (wide/medium/close)
(5) Background composition and depth
(6) Mood and atmospheric micro-details

Genre: {genre}
Premise: {premise}
Theme: {theme}
Do NOT violate: {negs}
Seed art direction (expand this intent): {seed}

Output ONLY the description. No headings. No JSON. No bullet labels."""

    try:
        out = generate_text(prompt, max_new_tokens=300, temperature=0.6)
        out = (out or "").strip()
        # Strip JSON artifacts
        out = re.sub(r"\{[^}]*\}", "", out, flags=re.DOTALL).strip()
        out = out.replace("```json", "").replace("```", "").strip()
        if len(out.split()) < 10:
            return seed.strip() + POSTER_SUFFIX
        return out + POSTER_SUFFIX
    except Exception as e:
        print(f"[B] Mistral failed ({e}), using seed fallback")
        return seed.strip() + POSTER_SUFFIX


art_b = {}
for cfg in idea_configs:
    art_b[cfg["label"]] = {}
    for vk in VARIANT_KEYS:
        print(f"[Approach B] {cfg['label']}_{vk}...")
        result = generate_extended_art_direction(
            cfg["movie_input"], cfg["genre"], cfg["variants"][vk]
        )
        art_b[cfg["label"]][vk] = result
        print(f"  → {len(result.split())} words: {result[:80]}...")


# --- Approach C: GPT-4o 6-layer expansion of Approach A ---
def enrich_with_gpt4o(art_direction: str, genre: str) -> str:
    client = OpenAI()
    user = f"""Rewrite this movie poster art direction into a detailed 150-200 word
Nano Banana image generation prompt with 6 layers:

LAYER 1 - SUBJECT: exact position, pose, expression, clothing
LAYER 2 - CINEMATIC STYLE: film stock (e.g. Kodak 5219, anamorphic lens, 35mm grain)
LAYER 3 - LIGHTING: source direction, Kelvin temperature, shadow hardness, rim light
LAYER 4 - COMPOSITION: shot type, camera angle, foreground/midground/background depth
LAYER 5 - COLOR PALETTE: exactly 3 named colors (e.g. "desaturated teal, burnt sienna, near-black charcoal")
LAYER 6 - ATMOSPHERE: time of day, weather, textures, emotional charge

Genre: {genre}
Original: {art_direction}

Output only the prompt text. No labels. No preamble."""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": "You are a professional cinematographer writing image generation prompts.",
                },
                {"role": "user", "content": user},
            ],
            temperature=0.7,
            max_tokens=400,
        )
        out = (resp.choices[0].message.content or "").strip()
        return out + POSTER_SUFFIX
    except Exception as e:
        print(f"[C] GPT-4o failed ({e}), using fallback")
        return art_direction.strip() + POSTER_SUFFIX


art_c = {}
for cfg in idea_configs:
    art_c[cfg["label"]] = {}
    for vk in VARIANT_KEYS:
        print(f"[Approach C] {cfg['label']}_{vk}...")
        result = enrich_with_gpt4o(cfg["variants"][vk], cfg["genre"])
        art_c[cfg["label"]][vk] = result
        print(f"  → {len(result.split())} words: {result[:80]}...")
        time.sleep(0.5)


# --- Print prompt comparison table ---
for cfg in idea_configs:
    for vk in VARIANT_KEYS:
        lab = cfg["label"]
        print(f"\n{'=' * 60}")
        print(f"{lab}_{vk}")
        print(f"{'=' * 60}")
        for key, d in [("A", art_a), ("B", art_b), ("C", art_c)]:
            text = d[lab][vk]
            print(f"  [{key}] {len(text.split())} words: {text[:120]}...")


# --- Image generation ---
def generate_one(idea_label, variant_key, approach_key, model_key, prompt) -> str:
    filename = f"{approach_key}_{model_key}_{idea_label}_{variant_key}.png"
    out_path = os.path.join(COMPARE_DIR, filename)

    if os.path.isfile(out_path) and not FORCE_REGENERATE:
        print(f"[{approach_key}/{model_key}] {idea_label}_{variant_key}: exists, skipping")
        return out_path
    if not (prompt or "").strip():
        print(f"[{approach_key}/{model_key}] {idea_label}_{variant_key}: empty prompt, skipping")
        return ""

    try:
        try:
            img_cfg = new_types.ImageConfig(aspect_ratio="9:16", image_size="1K")
        except Exception:
            img_cfg = new_types.ImageConfig(aspect_ratio="9:16")

        resp = nano_client.models.generate_content(
            model=MODELS[model_key],
            contents=prompt,
            config=new_types.GenerateContentConfig(
                response_modalities=["IMAGE"],
                image_config=img_cfg,
            ),
        )
        data = None
        for cand in getattr(resp, "candidates", []) or []:
            for part in getattr(getattr(cand, "content", None), "parts", []) or []:
                inline = getattr(part, "inline_data", None)
                if inline and getattr(inline, "data", None):
                    data = inline.data
                    break
            if data:
                break

        if not data:
            print(f"[{approach_key}/{model_key}] {idea_label}_{variant_key}: no image in response")
            return ""

        raw = data
        if isinstance(raw, str):
            import base64

            raw = base64.b64decode(raw)

        with open(out_path, "wb") as f:
            f.write(raw)
        print(f"[{approach_key}/{model_key}] {idea_label}_{variant_key}: ✅ saved")
        return out_path

    except Exception as e:
        print(f"[{approach_key}/{model_key}] {idea_label}_{variant_key}: ❌ {e}")
        return ""


approaches = {"A": art_a, "B": art_b, "C": art_c}
stats = {
    f"{a}_{m}": {"generated": 0, "skipped": 0, "failed": 0}
    for a in approaches
    for m in MODELS
}

for approach_key, art_dict in approaches.items():
    for model_key in MODELS:
        for cfg in idea_configs:
            for vk in VARIANT_KEYS:
                lab = cfg["label"]
                path = os.path.join(
                    COMPARE_DIR, f"{approach_key}_{model_key}_{lab}_{vk}.png"
                )
                existed = os.path.isfile(path)
                result = generate_one(lab, vk, approach_key, model_key, art_dict[lab][vk])
                key = f"{approach_key}_{model_key}"
                if existed and not FORCE_REGENERATE:
                    stats[key]["skipped"] += 1
                elif result and os.path.isfile(result):
                    stats[key]["generated"] += 1
                else:
                    stats[key]["failed"] += 1
                time.sleep(0.3)

print("\n=== Generation summary ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

# Store for Cell 8-J-2
COMPARE_APPROACHES = list(approaches.keys())
COMPARE_MODELS = list(MODELS.keys())
COMPARE_IDEAS = [cfg["label"] for cfg in idea_configs]
COMPARE_VARIANTS = VARIANT_KEYS
COMPARE_DIR_PATH = COMPARE_DIR


In [ ]:
# Cell 8-J-2: Display comparison grid

# Prerequisites: Cell 8-J-0 and Cell 8-J-1 must be fully executed first.
# Uses: COMPARE_APPROACHES, COMPARE_MODELS, COMPARE_IDEAS, COMPARE_VARIANTS, COMPARE_DIR_PATH
# Also uses: art_a, art_b, art_c, idea_configs

import matplotlib.pyplot as plt
import PIL.Image as PILImage
import pandas as pd
import os

# Layout:
# rows = ideas × variants (3 × 3 = 9 rows)
# cols = approaches × models (3 × 1or2 = 3 or 6 cols)

col_labels = [f"{a}_{m}" for a in COMPARE_APPROACHES for m in COMPARE_MODELS]
row_labels = [f"{idea}_{v}" for idea in COMPARE_IDEAS for v in COMPARE_VARIANTS]

n_rows = len(row_labels)  # 9
n_cols = len(col_labels)  # 3 or 6

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 6))

# Normalize axes to 2D array
if n_rows == 1:
    axes = [axes]
if n_cols == 1:
    axes = [[ax] for ax in axes]

for r, row_key in enumerate(row_labels):
    idea_label, variant_key = row_key.rsplit("_", 1)
    for c, col_key in enumerate(col_labels):
        approach_key, model_key = col_key.split("_", 1)
        ax = axes[r][c]
        path = os.path.join(
            COMPARE_DIR_PATH,
            f"{approach_key}_{model_key}_{idea_label}_{variant_key}.png",
        )
        if os.path.isfile(path):
            ax.imshow(PILImage.open(path))
        else:
            ax.text(
                0.5,
                0.5,
                "missing",
                ha="center",
                va="center",
                fontsize=8,
                color="gray",
            )
            ax.set_facecolor("#f0f0f0")
        ax.set_title(
            f"{approach_key}/{model_key}\n{idea_label} {variant_key}",
            fontsize=7,
        )
        ax.axis("off")

plt.suptitle(
    "Poster comparison: Approach (A=Mistral short / B=Mistral long / C=GPT-4o) × Model × Idea × Variant",
    y=1.005,
    fontsize=10,
)
plt.tight_layout()
grid_path = os.path.join(COMPARE_DIR_PATH, "comparison_grid.png")
plt.savefig(grid_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved: {grid_path}")

# Print prompt word count table
print("\n=== Prompt word counts per idea × variant ===")
header = f"{'label':<14} {'A words':<10} {'B words':<10} {'C words':<10}"
print(header)
print("-" * 46)
for cfg in idea_configs:
    lab = cfg["label"]
    for vk in COMPARE_VARIANTS:
        row_label = f"{lab}_{vk}"
        wa = len(art_a[lab][vk].split())
        wb = len(art_b[lab][vk].split())
        wc = len(art_c[lab][vk].split())
        print(f"{row_label:<14} {wa:<10} {wb:<10} {wc:<10}")

print("\n👆 Review grid. Decide which approach × model to use for Cell 8-H production run.")
print("   A = Mistral short (~60 words)")
print("   B = Mistral extended (~150 words)")
print("   C = GPT-4o 6-layer expanded (~180 words)")


## Section 9: QA Pair Generation & Validation

Cached artifacts live under **`to_be_cached/`** inside the cloned repo (same directory as `tmdb-fetch/`). Commit that folder after a full Colab run to skip expensive API calls next time. Delete a cached file to force regeneration.

| Cell | Description |
|------|-------------|
| 9-A | Generate 200 QA pairs (GPT-4o) → `to_be_cached/qa_pairs_raw.json` |
| 9-B | Mistral-7B critic → `to_be_cached/qa_pairs_validated.json` |
| 9-C | 80/20 split → `to_be_cached/qa_val.json`, `qa_test.json` |
| 9-D | Tuning grid (GPT judge) → `to_be_cached/tuning_results.csv` |

In [42]:
# Cell 9-A: QA pair generation with GPT-4o (reverse generation from TMDB metadata)

TOP_GENRES = ["Drama", "Action", "Thriller", "Comedy", "Horror"]


def _primary_top_genre(genre_names):
    """First matching label from TOP_GENRES (fixed order) for stratification."""
    if pd.isna(genre_names):
        return None
    s = str(genre_names).strip()
    # TMDB CSV stores genre_names as a Python-list string, e.g. "['Action', 'Thriller']"
    if s.startswith("[") and s.endswith("]"):
        gset_lower = {m.group(1).strip().lower() for m in re.finditer(r"['\"]([^'\"]*)['\"]", s)}
    else:
        # Comma-separated fallback
        gset_lower = {x.strip().lower() for x in s.split(",") if x.strip()}
    for g in TOP_GENRES:
        if g.lower() in gset_lower:
            return g
    return None


def _stratified_sample_movies(df: pd.DataFrame, strat_col: str, n: int, random_state: int = 42) -> pd.DataFrame:
    """Sample n rows stratified by strat_col using a multinomial allocation (no sklearn)."""
    rng = np.random.RandomState(random_state)
    work = df[df[strat_col].notna()].copy()
    vc = work[strat_col].value_counts().sort_index()
    if vc.empty:
        raise ValueError("No rows with stratification labels")
    props = (vc / vc.sum()).values
    draws = rng.multinomial(n, props)
    parts = []
    for label, draw in zip(vc.index, draws):
        sub = work[work[strat_col] == label]
        use = min(int(draw), len(sub))
        if use > 0:
            parts.append(sub.sample(n=use, random_state=rng.randint(0, 2**31 - 1)))
    result = pd.concat(parts) if parts else work.iloc[:0]
    if len(result) < n:
        avail = work.loc[~work.index.isin(result.index)]
        need = n - len(result)
        if len(avail) >= need:
            extra = avail.sample(n=need, random_state=rng.randint(0, 2**31 - 1))
            result = pd.concat([result, extra])
    if len(result) > n:
        result = result.sample(n=n, random_state=random_state)
    return result.reset_index(drop=True)


# API outputs are stored under to_be_cached/ — if present, skip GPT-4o calls.
CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_RAW_CACHE = os.path.join(CACHE_DIR, "qa_pairs_raw.json")

if os.path.isfile(QA_RAW_CACHE):
    with open(QA_RAW_CACHE, "r", encoding="utf-8") as f:
        qa_pairs_raw = json.load(f)
    print(f"Loaded {len(qa_pairs_raw)} QA pairs from cache (skipped GPT-4o): {QA_RAW_CACHE}")
else:
    qa_gen_client = OpenAI()

    sample_df = corpus_df.copy()
    sample_df["_stratum"] = sample_df["genre_names"].map(_primary_top_genre)
    sample_200 = _stratified_sample_movies(sample_df, "_stratum", 200, random_state=42)

    qa_pairs_raw = []
    SYSTEM_QA = (
        "You reverse-engineer a structured creative brief from movie metadata. "
        "Output a single JSON object only. Do not copy or closely paraphrase the tagline in any field."
    )

    for _, row in tqdm(sample_200.iterrows(), total=len(sample_200), desc="GPT-4o QA generation"):
        movie_id = int(row["id"])
        title = str(row["title"])
        tagline = str(row["tagline"]) if pd.notna(row["tagline"]) else ""
        overview = str(row["overview"]) if pd.notna(row["overview"]) else ""
        genre_names = str(row["genre_names"]) if pd.notna(row["genre_names"]) else ""

        user_msg = f"""From this TMDB movie metadata, produce a JSON object with exactly these keys:
- core_premise: one sentence setting and situation (no invented plot specifics not implied below)
- thematic_core: emotional or psychological theme
- negative_constraints: explicit comma-separated list of elements NOT to invent later

Metadata:
title: {title}
tagline: {tagline}
overview: {overview}
genre_names: {genre_names}
"""

        try:
            resp = qa_gen_client.chat.completions.create(
                model="gpt-4o",
                response_format={"type": "json_object"},
                temperature=0.5,
                messages=[
                    {"role": "system", "content": SYSTEM_QA},
                    {"role": "user", "content": user_msg},
                ],
            )
            payload = json.loads(resp.choices[0].message.content.strip())
            qa_pairs_raw.append(
                {
                    "movie_id": movie_id,
                    "title": title,
                    "genre_names": genre_names,
                    "core_premise": str(payload.get("core_premise", "")).strip(),
                    "thematic_core": str(payload.get("thematic_core", "")).strip(),
                    "negative_constraints": str(payload.get("negative_constraints", "")).strip(),
                    "gold_tagline": tagline,
                    "gold_overview": overview,
                }
            )
        except Exception as e:
            print(f"[skip movie_id={movie_id}] {e}")

        time.sleep(0.3)

    with open(QA_RAW_CACHE, "w", encoding="utf-8") as f:
        json.dump(qa_pairs_raw, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(qa_pairs_raw)} records to {QA_RAW_CACHE}")

Loaded 200 QA pairs from cache (skipped GPT-4o): ryuichi-github-info290-2026s-group5/to_be_cached/qa_pairs_raw.json


In [43]:
# Cell 9-B: LLM critic validation with Mistral-7B

CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_RAW_CACHE = os.path.join(CACHE_DIR, "qa_pairs_raw.json")
QA_VALIDATED_CACHE = os.path.join(CACHE_DIR, "qa_pairs_validated.json")
LEGACY_RAW = os.path.join(REPO_ROOT, "qa_pairs_raw.json")


def _extract_critic_json(text: str) -> dict:
    """Brace-scan JSON extraction (same idea as extract_best_json) for critic schema."""
    decoder = json.JSONDecoder()
    i = 0
    best = None
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            if isinstance(obj, dict) and "premise_faithful" in obj and "constraints_specific" in obj:
                best = obj
            i = start + end
        except json.JSONDecodeError:
            i = start + 1
    if best is None:
        raise ValueError("No critic JSON object found in model output.")
    return best


def _critic_prompt(rec: dict) -> str:
    return f"""You are a strict QA validator for reverse-generated movie briefs.

Movie ID: {rec["movie_id"]}
Title: {rec["title"]}
Original overview (ground truth for what the movie is about):
{rec["gold_overview"]}

Generated fields to check:
core_premise: {rec["core_premise"]}
thematic_core: {rec["thematic_core"]}
negative_constraints: {rec["negative_constraints"]}

Tasks:
1) Does core_premise faithfully reflect the movie overview without inventing characters, events, or settings not supported by the overview?
2) Does negative_constraints list at least two concrete elements that are clearly absent from core_premise (not vague words like "nothing")?

Reply with ONE JSON object only, keys:
"movie_id" (int), "premise_faithful" (bool), "constraints_specific" (bool), "reason" (one short sentence).
No markdown, no text outside JSON.
"""


if os.path.isfile(QA_VALIDATED_CACHE):
    with open(QA_VALIDATED_CACHE, "r", encoding="utf-8") as f:
        validated = json.load(f)
    print(f"Loaded {len(validated)} validated QA pairs from cache (skipped Mistral): {QA_VALIDATED_CACHE}")
else:
    raw_path = QA_RAW_CACHE if os.path.isfile(QA_RAW_CACHE) else LEGACY_RAW
    if not os.path.isfile(raw_path):
        raise FileNotFoundError(f"Need {QA_RAW_CACHE} (run 9-A) or legacy {LEGACY_RAW}")
    with open(raw_path, "r", encoding="utf-8") as f:
        qa_raw = json.load(f)

    validated = []
    failed = 0

    for rec in tqdm(qa_raw, desc="Mistral critic"):
        prompt = _critic_prompt(rec)
        try:
            raw_out = generate_text(prompt, max_new_tokens=256)
            parsed = _extract_critic_json(raw_out)
            pf = bool(parsed.get("premise_faithful", False))
            cs = bool(parsed.get("constraints_specific", False))
            reason = str(parsed.get("reason", "")).strip()
            if pf and cs:
                out = dict(rec)
                out["critic_reason"] = reason
                validated.append(out)
            else:
                failed += 1
        except Exception as e:
            failed += 1
            print(f"[critic fail movie_id={rec.get('movie_id')}] {e}")

    print(f"Passed: {len(validated)}  Failed: {failed}")

    with open(QA_VALIDATED_CACHE, "w", encoding="utf-8") as f:
        json.dump(validated, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(validated)} validated records to {QA_VALIDATED_CACHE}")

Loaded 181 validated QA pairs from cache (skipped Mistral): ryuichi-github-info290-2026s-group5/to_be_cached/qa_pairs_validated.json


In [44]:
# Cell 9-C: val / test split (80/20, stratified by primary top-5 genre)

CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_VALIDATED_CACHE = os.path.join(CACHE_DIR, "qa_pairs_validated.json")
QA_VAL_CACHE = os.path.join(CACHE_DIR, "qa_val.json")
QA_TEST_CACHE = os.path.join(CACHE_DIR, "qa_test.json")
LEGACY_VALIDATED = os.path.join(REPO_ROOT, "qa_pairs_validated.json")


def _stratified_train_test(df: pd.DataFrame, strat_col: str, test_size: float = 0.2, random_state: int = 42):
    rng = np.random.RandomState(random_state)
    train_idx = []
    test_idx = []
    for _, sub in df.groupby(strat_col):
        idx = sub.index.to_numpy()
        rng.shuffle(idx)
        n = len(idx)
        n_test = int(round(n * test_size))
        if n <= 1:
            train_idx.extend(idx.tolist())
        else:
            n_test = max(1, min(n_test, n - 1))
            test_idx.extend(idx[:n_test].tolist())
            train_idx.extend(idx[n_test:].tolist())
    return df.loc[train_idx], df.loc[test_idx]


if os.path.isfile(QA_VAL_CACHE) and os.path.isfile(QA_TEST_CACHE):
    with open(QA_VAL_CACHE, "r", encoding="utf-8") as f:
        qa_val_records = json.load(f)
    with open(QA_TEST_CACHE, "r", encoding="utf-8") as f:
        qa_test_records = json.load(f)
    print(f"Loaded val/test from cache (skipped split): {QA_VAL_CACHE}, {QA_TEST_CACHE}")
    qa_val_df = pd.DataFrame(qa_val_records)
    qa_test_df = pd.DataFrame(qa_test_records)
    qa_val_df["_stratum"] = qa_val_df["genre_names"].map(_primary_top_genre)
    qa_test_df["_stratum"] = qa_test_df["genre_names"].map(_primary_top_genre)
else:
    val_in = QA_VALIDATED_CACHE if os.path.isfile(QA_VALIDATED_CACHE) else LEGACY_VALIDATED
    if not os.path.isfile(val_in):
        raise FileNotFoundError(f"Need {QA_VALIDATED_CACHE} (run 9-B) or legacy {LEGACY_VALIDATED}")
    with open(val_in, "r", encoding="utf-8") as f:
        qa_validated = json.load(f)

    qa_df = pd.DataFrame(qa_validated)
    qa_df["_stratum"] = qa_df["genre_names"].map(_primary_top_genre)
    qa_strat = qa_df[qa_df["_stratum"].notna()].copy()
    if len(qa_strat) < len(qa_df):
        print(f"Dropped {len(qa_df) - len(qa_strat)} rows without a top-5 primary genre for stratification")

    qa_val_df, qa_test_df = _stratified_train_test(qa_strat, "_stratum", test_size=0.2, random_state=42)

    qa_val_records = qa_val_df.drop(columns=["_stratum"], errors="ignore").to_dict(orient="records")
    qa_test_records = qa_test_df.drop(columns=["_stratum"], errors="ignore").to_dict(orient="records")

    with open(QA_VAL_CACHE, "w", encoding="utf-8") as f:
        json.dump(qa_val_records, f, ensure_ascii=False, indent=2)
    with open(QA_TEST_CACHE, "w", encoding="utf-8") as f:
        json.dump(qa_test_records, f, ensure_ascii=False, indent=2)

print(f"Val size: {len(qa_val_records)}  Test size: {len(qa_test_records)}")
print("Val genre distribution (primary top-5):")
print(qa_val_df["_stratum"].value_counts())
print("Test genre distribution (primary top-5):")
print(qa_test_df["_stratum"].value_counts())

Loaded val/test from cache (skipped split): ryuichi-github-info290-2026s-group5/to_be_cached/qa_val.json, ryuichi-github-info290-2026s-group5/to_be_cached/qa_test.json
Val size: 145  Test size: 36
Val genre distribution (primary top-5):
_stratum
Drama       66
Action      38
Comedy      26
Thriller     9
Horror       6
Name: count, dtype: int64
Test genre distribution (primary top-5):
_stratum
Drama       16
Action       9
Comedy       7
Horror       2
Thriller     2
Name: count, dtype: int64


In [45]:
# Cell 9-D: Hyperparameter tuning on val subset (llm_as_judge on V2 output)
# -----------------------------------------------------------------------
# SMOKE-TEST config (n=3, 2×2×2 grid) — for quick validation only.
# For production run, set SMOKE_TEST = False.
# Smoke-test results are saved to tuning_results_smoke.csv (NOT tuning_results.csv)
# to avoid polluting the production cache.
# -----------------------------------------------------------------------

SMOKE_TEST = False  # ← Set False for production (n=20, 3×2×3 grid, ~5 hours)

CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_VAL_CACHE = os.path.join(CACHE_DIR, "qa_val.json")
LEGACY_QA_VAL = os.path.join(REPO_ROOT, "qa_val.json")

# Cache file differs by mode — smoke-test results never pollute production cache
TUNING_CACHE = os.path.join(CACHE_DIR,
    "tuning_results_smoke.csv" if SMOKE_TEST else "tuning_results.csv")

if os.path.isfile(TUNING_CACHE):
    tuning_df = pd.read_csv(TUNING_CACHE)
    mode_label = "smoke-test" if SMOKE_TEST else "production"
    print(f"Loaded {mode_label} tuning results from cache: {TUNING_CACHE}")
else:
    val_path = QA_VAL_CACHE if os.path.isfile(QA_VAL_CACHE) else LEGACY_QA_VAL
    if not os.path.isfile(val_path):
        raise FileNotFoundError(f"Need {QA_VAL_CACHE} (run 9-C) or legacy {LEGACY_QA_VAL}")
    with open(val_path, "r", encoding="utf-8") as f:
        qa_val_list = json.load(f)

    # Config differs by mode
    if SMOKE_TEST:
        n_records = 3
        grid_k    = [3, 5]
        grid_gf   = [True, False]
        grid_temp = [0.4, 0.6]
        print(f"Running SMOKE-TEST: n={n_records}, grid={len(grid_k)}×{len(grid_gf)}×{len(grid_temp)}={len(grid_k)*len(grid_gf)*len(grid_temp)} configs")
    else:
        n_records = 20
        grid_k    = [3, 5, 7]
        grid_gf   = [True, False]
        grid_temp = [0.4, 0.6, 0.8]
        print(f"Running PRODUCTION: n={n_records}, grid={len(grid_k)}×{len(grid_gf)}×{len(grid_temp)}={len(grid_k)*len(grid_gf)*len(grid_temp)} configs (~5 hours)")

    qa_val_tune = pd.DataFrame(qa_val_list).sample(n=min(n_records, len(qa_val_list)), random_state=42)
    tune_records = qa_val_tune.to_dict(orient="records")

    judge_client_tune = OpenAI()
    rows = []

    for k in grid_k:
        for use_gf in grid_gf:
            for temperature in grid_temp:
                scores = []
                n_ok = 0
                for rec in tune_records:
                    movie_input = {
                        "core_premise":        rec["core_premise"],
                        "thematic_core":       rec["thematic_core"],
                        "negative_constraints": rec["negative_constraints"],
                    }
                    if use_gf:
                        gn = str(rec.get("genre_names", "") or "")
                        # Fix: parse Python-list string e.g. "['Action','Thriller']"
                        gf = [m.group(1).strip().lower()
                              for m in re.finditer(r"['\"]([^'\"]*)['\"]", gn)]
                        genre_filter = gf if gf else None
                    else:
                        genre_filter = None
                    try:
                        _v1, v2, _topk, _refs = run_v1_v2(
                            movie_input, k=k, genre_filter=genre_filter, temperature=temperature
                        )
                        judge_out = llm_as_judge(
                            movie_input, v2, client=judge_client_tune, verbose=False
                        )
                        if judge_out.get("error"):
                            continue
                        scores.append(float(judge_out["average"]))
                        n_ok += 1
                    except Exception as e:
                        print(f"[tune skip movie_id={rec.get('movie_id')}] {e}")

                avg_score = float(np.mean(scores)) if scores else float("nan")
                rows.append({
                    "k":            k,
                    "genre_filter": use_gf,
                    "temperature":  temperature,
                    "avg_score":    round(avg_score, 3) if not np.isnan(avg_score) else float("nan"),
                    "n_evaluated":  n_ok,
                })
                print(f"  k={k} gf={use_gf} temp={temperature} → avg={avg_score:.3f} (n={n_ok})")

    tuning_df = pd.DataFrame(rows)
    tuning_df.to_csv(TUNING_CACHE, index=False)
    print(f"\nWrote {TUNING_CACHE}")

print("\nTop 3 configs by avg_score:")
print(
    tuning_df.sort_values("avg_score", ascending=False, na_position="last")
    .head(3)
    .to_string(index=False)
)

Running PRODUCTION: n=20, grid=3×2×3=18 configs (~5 hours)
  k=3 gf=True temp=0.4 → avg=4.430 (n=20)
  k=3 gf=True temp=0.6 → avg=4.430 (n=20)
  k=3 gf=True temp=0.8 → avg=4.410 (n=20)
  k=3 gf=False temp=0.4 → avg=4.410 (n=20)
  k=3 gf=False temp=0.6 → avg=4.500 (n=20)
[tune skip movie_id=833425] Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}
  k=3 gf=False temp=0.8 → avg=4.453 (n=19)
  k=5 gf=True temp=0.4 → avg=4.410 (n=20)
  k=5 gf=True temp=0.6 → avg=4.420 (n=20)
  k=5 gf=True temp=0.8 → avg=4.410 (n=20)
  k=5 gf=False temp=0.4 → avg=4.450 (n=20)
  k=5 gf=False temp=0.6 → avg=4.380 (n=20)
  k=5 gf=False temp=0.8 → avg

In [46]:
# Cell 9-E: Final evaluation on test set (V1 vs V2 vs V3)

# Smoke mode runs 3 records only; production runs the full qa_test split (~36 records).
SMOKE_TEST = False  # Set False for production (all 36 test records, ~1.5 hours)
mode_label = "SMOKE-TEST" if SMOKE_TEST else "PRODUCTION"

TEST_RESULTS_CACHE = os.path.join(
    CACHE_DIR,
    "test_results_smoke.json" if SMOKE_TEST else "test_results.json",
)

TUNING_CACHE = os.path.join(CACHE_DIR,
    "tuning_results_smoke.csv" if SMOKE_TEST else "tuning_results.csv")
if not os.path.isfile(TUNING_CACHE):
    raise FileNotFoundError(
        f"tuning_results.csv not found at {TUNING_CACHE}. Run Cell 9-D first."
    )
tuning_df = pd.read_csv(TUNING_CACHE)

best = tuning_df.sort_values("avg_score", ascending=False).iloc[0]
best_k = int(best["k"])
best_temp = float(best["temperature"])
best_gf = bool(best["genre_filter"])

QA_TEST_PATH = os.path.join(CACHE_DIR, "qa_test.json")
if not os.path.isfile(QA_TEST_PATH):
    raise FileNotFoundError(
        f"qa_test.json not found at {QA_TEST_PATH}. Run Cell 9-C to create the val/test split."
    )
with open(QA_TEST_PATH, "r", encoding="utf-8") as f:
    qa_test_records = json.load(f)


def _genre_filter_from_record(rec, use_gf):
    if not use_gf:
        return None
    gn = str(rec.get("genre_names", "") or "")
    s = gn.strip()
    if s.startswith("[") and s.endswith("]"):
        gf = [m.group(1).strip().lower() for m in re.finditer(r"['\"]([^'\"]*)['\"]", s)]
    else:
        gf = [x.strip().lower() for x in s.split(",") if x.strip()]
    gf = [x for x in gf if x]
    return gf if gf else None


if SMOKE_TEST:
    n_records = 3
    print(f"[{mode_label}] Running SMOKE-TEST: n={n_records} records")
else:
    n_records = len(qa_test_records)
    print(
        f"[{mode_label}] Running PRODUCTION: n={n_records} records (~1.5 hours)"
    )

qa_test_run = qa_test_records[:n_records]

if os.path.isfile(TEST_RESULTS_CACHE):
    with open(TEST_RESULTS_CACHE, "r", encoding="utf-8") as f:
        cached = json.load(f)
    print(f"[{mode_label}] Loaded from cache: {TEST_RESULTS_CACHE}")
    test_results_v1 = cached["v1"]
    test_results_v2 = cached["v2"]
    test_results_v3 = cached["v3"]
    summary_v1 = cached["summary"]["v1"]
    summary_v2 = cached["summary"]["v2"]
    summary_v3 = cached["summary"]["v3"]
else:
    judge_client = OpenAI()

    test_results_v1 = []
    test_results_v2 = []
    test_results_v3 = []

    for i, rec in enumerate(qa_test_run, start=1):
        print(f"[{mode_label}] Evaluating record {i}/{n_records}...")
        movie_input = {
            "core_premise": rec["core_premise"],
            "thematic_core": rec["thematic_core"],
            "negative_constraints": rec["negative_constraints"],
        }
        genre_filter = _genre_filter_from_record(rec, best_gf)

        v1, v2, _topk, _refs = run_v1_v2(
            movie_input,
            k=best_k,
            genre_filter=genre_filter,
            temperature=best_temp,
        )
        v3, _v3_hist = run_v3(
            movie_input,
            k=best_k,
            genre_filter=genre_filter,
            temperature=best_temp,
            judge_client=judge_client,
        )

        movie_id = int(rec["movie_id"])
        judge_out_v1 = llm_as_judge(movie_input, v1, client=judge_client, verbose=False)
        judge_out_v2 = llm_as_judge(movie_input, v2, client=judge_client, verbose=False)
        judge_out_v3 = llm_as_judge(movie_input, v3, client=judge_client, verbose=False)

        test_results_v1.append({"id": movie_id, "result": judge_out_v1})
        test_results_v2.append({"id": movie_id, "result": judge_out_v2})
        test_results_v3.append({"id": movie_id, "result": judge_out_v3})

        if i < n_records:
            time.sleep(1.0)

    summary_v1 = summarize_batch(test_results_v1)
    summary_v2 = summarize_batch(test_results_v2)
    summary_v3 = summarize_batch(test_results_v3)

    with open(TEST_RESULTS_CACHE, "w", encoding="utf-8") as f:
        json.dump(
            {
                "v1": test_results_v1,
                "v2": test_results_v2,
                "v3": test_results_v3,
                "summary": {"v1": summary_v1, "v2": summary_v2, "v3": summary_v3},
                "smoke_test": SMOKE_TEST,
                "n_evaluated": n_records,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )
    print(f"[{mode_label}] Wrote {TEST_RESULTS_CACHE}")

summary_rows = []
for variant, s in [("V1", summary_v1), ("V2", summary_v2), ("V3", summary_v3)]:
    row = {"variant": variant, "overall_mean": s["overall_mean"]}
    for dim in DIMENSIONS:
        row[dim] = s["per_dimension"].get(dim, {}).get("mean")
    summary_rows.append(row)

summary_compare_df = pd.DataFrame(summary_rows)[["variant", "overall_mean"] + list(DIMENSIONS)]
print(f"\n[{mode_label}] === Test set summary (V1 vs V2 vs V3) ===\n")
print(summary_compare_df.to_string(index=False))

print(f"[{mode_label}] ✅ Final evaluation complete. Results saved.")


[PRODUCTION] Running PRODUCTION: n=36 records (~1.5 hours)
[PRODUCTION] Evaluating record 1/36...
[v3 iter 1] tagline="Confronting the ghosts of a revolutionist's past." average=4.20
[PRODUCTION] Evaluating record 2/36...
[v3 iter 1] tagline='Fighting for life amidst chaos.' average=4.20
[PRODUCTION] Evaluating record 3/36...
[v3 iter 1] tagline='Duty calls the unwilling.' average=4.00
[PRODUCTION] Evaluating record 4/36...
[v3 iter 1] tagline='Unlikely hero, unlikely victory.' average=4.60
[PRODUCTION] Evaluating record 5/36...
[v3 iter 1] tagline='This Christmas, peace is not a given.' average=4.40
[PRODUCTION] Evaluating record 6/36...
[v3 iter 1] tagline='Love, a weapon unyielding.' average=4.80
[PRODUCTION] Evaluating record 7/36...
[v3 iter 1] tagline='Family bonding or global saving?' average=4.60
[PRODUCTION] Evaluating record 8/36...
[v3 iter 1] tagline='Weapons forged in fire. Justice tempered in steel.' average=5.00
[PRODUCTION] Evaluating record 9/36...
[v3 iter 1] tagline=

In [47]:
# Cell 10-A: Build and publish all Web UI assets to docs/
# Prerequisites: Run Cell 9-E first (sets summary_v1/v2/v3, SMOKE_TEST, n_records).
#                Run Cell 8-G and 8-H first (sets v1/v2/v3 for all ideas, poster images).
#                Place index.html in REPO_ROOT before running.

import shutil
import subprocess

DOCS_DIR      = os.path.join(REPO_ROOT, "docs")
DOCS_DATA_DIR = os.path.join(DOCS_DIR, "data")
RESULTS_JSON  = os.path.join(DOCS_DATA_DIR, "results.json")
POSTER_SRC    = os.path.join(REPO_ROOT, "poster_images")
POSTER_DST    = os.path.join(DOCS_DIR, "poster_images")
INDEX_SRC     = os.path.join(REPO_ROOT, "index.html")
INDEX_DST     = os.path.join(DOCS_DIR, "index.html")

os.makedirs(DOCS_DATA_DIR, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────

def _scores(result):
    dims = ["narrative_fidelity", "genre_alignment", "visual_specificity",
            "creative_specificity", "output_format_validity"]
    s = {d: result["scores"][d]["score"] for d in dims}
    s["average"] = result["average"]
    return s

def _trend(v1_avg, v2_avg, v3_avg):
    """Return ordering string e.g. 'V1<V2<V3', 'V2=V3<V1'."""
    pairs = sorted([("V1", v1_avg), ("V2", v2_avg), ("V3", v3_avg)], key=lambda x: x[1])
    parts, i = [], 0
    while i < len(pairs):
        group = [pairs[i][0]]
        while i + 1 < len(pairs) and pairs[i+1][1] == pairs[i][1]:
            i += 1
            group.append(pairs[i][0])
        parts.append("=".join(group))
        i += 1
    return "<".join(parts)

def _git_push(repo_root, message):
    subprocess.run(["git", "-C", repo_root, "add", "docs/"], check=True)
    r = subprocess.run(
        ["git", "-C", repo_root, "commit", "-m", message],
        capture_output=True, text=True
    )
    if r.returncode != 0 and "nothing to commit" in (r.stdout + r.stderr):
        print("  Nothing new to commit.")
        return
    subprocess.run(["git", "-C", repo_root, "push"], check=True)
    print(f"  ✅ Pushed: {message}")

# ── Step 1: Build ideas list ──────────────────────────────────────────

idea_configs = [
    {
        "id": 1, "genre": "Psychological Thriller",
        "movie_input": movie_input,
        "variants": [
            (v1_idea1, result_v1,       "V1", "Option A", "Baseline: no retrieval, no visual grounding",    "poster_images/idea1_v1.png"),
            (v2_idea1, result_v2,       "V2", "Option B", "RAG retrieval + poster caption context",         "poster_images/idea1_v2.png"),
            (v3_idea1, result_v3,       "V3", "Option C", "V2 + iterative LLM-as-Judge refinement",         "poster_images/idea1_v3.png"),
        ]
    },
    {
        "id": 2, "genre": "Drama — Family & Loss",
        "movie_input": movie_input_idea2,
        "variants": [
            (v1_idea2, result_v1_idea2, "V1", "Option A", "Baseline: no retrieval, no visual grounding",    "poster_images/idea2_v1.png"),
            (v2_idea2, result_v2_idea2, "V2", "Option B", "RAG retrieval + poster caption context",         "poster_images/idea2_v2.png"),
            (v3_idea2, result_v3_idea2, "V3", "Option C", "V2 + iterative LLM-as-Judge refinement",         "poster_images/idea2_v3.png"),
        ]
    },
    {
        "id": 3, "genre": "Action Thriller",
        "movie_input": movie_input_idea3,
        "variants": [
            (v1_idea3, result_v1_idea3, "V1", "Option A", "Baseline: no retrieval, no visual grounding",    "poster_images/idea3_v1.png"),
            (v2_idea3, result_v2_idea3, "V2", "Option B", "RAG retrieval + poster caption context",         "poster_images/idea3_v2.png"),
            (v3_idea3, result_v3_idea3, "V3", "Option C", "V2 + iterative LLM-as-Judge refinement",         "poster_images/idea3_v3.png"),
        ]
    },
]

ideas = []
for cfg in idea_configs:
    mi = cfg["movie_input"]
    variants = []
    for (output, result, label, blind, desc, img) in cfg["variants"]:
        variants.append({
            "label":      label,
            "blind":      blind,
            "desc":       desc,
            "tagline":    output.get("tagline", ""),
            "overview":   output.get("overview", ""),
            "poster_art": output.get("poster_art_direction", ""),
            "image":      img,
            "scores":     _scores(result),
        })
    ideas.append({
        "id":           cfg["id"],
        "genre":        cfg["genre"],
        "premise":      mi.get("core_premise", ""),
        "thematic_core": mi.get("thematic_core", ""),
        "variants":     variants,
    })

print(f"✅ Ideas built: {len(ideas)} ideas, {sum(len(i['variants']) for i in ideas)} variants")

# ── Step 2: Cross-idea summary ────────────────────────────────────────

cross_idea_summary = []
for cfg, idea in zip(idea_configs, ideas):
    avgs = {v["label"]: v["scores"]["average"] for v in idea["variants"]}
    cross_idea_summary.append({
        "idea":   f"Idea {cfg['id']}",
        "V1_avg": avgs["V1"],
        "V2_avg": avgs["V2"],
        "V3_avg": avgs["V3"],
        "trend":  _trend(avgs["V1"], avgs["V2"], avgs["V3"]),
    })

print(f"✅ Cross-idea summary built")

# ── Step 3: test_set_summary from Cell 9-E ────────────────────────────

test_set_summary = {
    "smoke_test":  SMOKE_TEST,
    "n_evaluated": n_records,
    "v1": {"overall_mean": summary_v1["overall_mean"], "per_dimension": summary_v1["per_dimension"]},
    "v2": {"overall_mean": summary_v2["overall_mean"], "per_dimension": summary_v2["per_dimension"]},
    "v3": {"overall_mean": summary_v3["overall_mean"], "per_dimension": summary_v3["per_dimension"]},
}

print(f"✅ test_set_summary built (smoke={SMOKE_TEST}, n={n_records})")

# ── Step 4: Write results.json ────────────────────────────────────────

results = {
    "ideas":               ideas,
    "cross_idea_summary":  cross_idea_summary,
    "test_set_summary":    test_set_summary,
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ results.json written → {RESULTS_JSON}")

# ── Step 5: Copy poster_images into docs/ ─────────────────────────────

if os.path.exists(POSTER_DST):
    shutil.rmtree(POSTER_DST)
shutil.copytree(POSTER_SRC, POSTER_DST)
n_images = len([f for f in os.listdir(POSTER_DST) if f.endswith(".png")])
print(f"✅ poster_images copied: {n_images} images → {POSTER_DST}")

# ── Step 6: Copy index.html into docs/ ───────────────────────────────

if os.path.isfile(INDEX_SRC):
    shutil.copy(INDEX_SRC, INDEX_DST)
    print(f"✅ index.html copied → {INDEX_DST}")
else:
    print(f"⚠ index.html not found at {INDEX_SRC}")
    print("  Upload index.html to REPO_ROOT and re-run this cell.")

# ── Step 7: Commit and push docs/ ────────────────────────────────────

mode = "smoke" if SMOKE_TEST else "production"
_git_push(
    REPO_ROOT,
    f"update: Web UI assets — results.json + {n_images} images ({mode}, n_test={n_records})"
)

# ── Step 8: Final summary ─────────────────────────────────────────────

print()
print("=" * 60)
print("CELL 10-A COMPLETE")
print("=" * 60)
print(f"  results.json : {len(ideas)} ideas / {sum(len(i['variants']) for i in ideas)} variants")
print(f"  poster images: {n_images}")
print(f"  test results : {'smoke-test' if SMOKE_TEST else 'production'} (n={n_records})")
print(f"  index.html   : {'✅ copied' if os.path.isfile(INDEX_DST) else '⚠ missing — upload manually'}")
print()
print("  Next: GitHub Settings → Pages → Source: main / docs/")
print("  URL : https://ryuichi-github.github.io/info290-2026s-group5/")
print("=" * 60)

✅ Ideas built: 3 ideas, 9 variants
✅ Cross-idea summary built
✅ test_set_summary built (smoke=False, n=36)
✅ results.json written → ryuichi-github-info290-2026s-group5/docs/data/results.json
✅ poster_images copied: 9 images → ryuichi-github-info290-2026s-group5/docs/poster_images
⚠ index.html not found at ryuichi-github-info290-2026s-group5/index.html
  Upload index.html to REPO_ROOT and re-run this cell.
  ✅ Pushed: update: Web UI assets — results.json + 9 images (production, n_test=36)

CELL 10-A COMPLETE
  results.json : 3 ideas / 9 variants
  poster images: 9
  test results : production (n=36)
  index.html   : ⚠ missing — upload manually

  Next: GitHub Settings → Pages → Source: main / docs/
  URL : https://ryuichi-github.github.io/info290-2026s-group5/
